# RMIT_SGS_Sigma69_C1 — V5, nền V3 và thử hiệu chỉnh Portuguese

File đang chạy: **`final_97.ipynb`**. Wolf Defender/ModernBERT được fine-tune toàn bộ,
nguồn Kaggle đã chọn: `aisuko/wolf-defender-prompt-injection/transformers/default/1`.

V3 có raw validation Macro F1 **0.972933**, tốt nhất ở epoch 4.
V4 người dùng báo **0.969274**, tốt nhất ở epoch 3. V4 đổi context, batch và cách
nạp config cùng lúc; các số này không xác định riêng nguyên nhân giảm điểm.

V5 trở lại cấu hình train và cách nạp V3: **256 token đầu, batch 8 × accumulation 4,
4 epoch, LR 3e-5, weighted CE power 0.5**. Thử thêm một hiệu chỉnh nhỏ cho các
dự đoán Portuguese BENIGN/JAILBREAK sát nhau. Chỉ bật nếu nested cross-fit qua
điều kiện; nếu không, giữ kết quả global ban đầu. **Chưa có điểm GPU/Kaggle V5.**

Upload notebook và chọn GPU mới, chạy từ đầu. Mục 3 nhận `rmit-hackathon-2026.zip`.
Chỉ còn `submission.csv` cũ thì cần train lại: CSV nhãn không chứa logits/validation.
Mặc định `USE_FULL_TRAIN=False` để xem selection trước. Khi đã xem kết quả và muốn
refit, bật tùy chọn này cho bước 11; không đổi các cấu hình khác giữa hai bước.

Sau mục 10 tải **`submission_selection.csv` và `selection_predictions.zip`**.
ZIP nhỏ giữ logits/ID/báo cáo để lần sau thử hậu xử lý mà không phải train lại.
`rmit_c1_artifacts.zip` cuối notebook mới chứa cả model. Mục tiêu 0.99 cần đo thực tế.


## 1. Cài dependencies

Giữ bộ Transformers/PEFT/bitsandbytes của lần chạy cũ và pin scikit-learn theo baseline.
Nếu cell yêu cầu restart, chỉ restart sau khi đã tải/lưu artifacts của phiên cũ; chạy lại từ đầu.


In [ ]:
import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

PINS = {
    "transformers": "4.57.6",
    "peft": "0.17.1",
    "accelerate": "1.10.1",
    "bitsandbytes": "0.48.1",
    "scikit-learn": "1.6.1",
}
loaded_conflicts = []
for package, wanted in PINS.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    module_name = "sklearn" if package == "scikit-learn" else package
    if module_name in sys.modules and installed != wanted:
        loaded_conflicts.append(package)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    *[f"{name}=={release}" for name, release in PINS.items()],
    "huggingface-hub>=0.34,<1",
    "pandas>=2.2,<3", "joblib>=1.2,<2",
    "matplotlib>=3.8,<4", "safetensors>=0.4.3", "kagglehub>=0.3.13,<1",
])
if loaded_conflicts:
    raise RuntimeError(
        "Da doi thu vien dang duoc import: " + ", ".join(loaded_conflicts)
        + ". Chon Runtime > Restart session, sau do chay lai tu dau."
    )
print("Dependencies ready. PyTorch CUDA cua Colab duoc giu nguyen.")


## 2. Cấu hình

Giữ `DATA_MODE="upload"` và chọn ZIP dữ liệu. Dữ liệu đã có trên Colab/Drive thì
dùng `"path"` và sửa `DATA_DIR`; `OUTPUT_ROOT` trên Drive giúp giữ checkpoint.

Wolf mặc định khôi phục cấu hình V3: 256 token `head`, batch 8 × accumulation 4 = 32.
Tự chọn epoch có Macro F1 cao nhất; không ép epoch 3 (V3 tốt nhất ở epoch 4).
Giữ class-weight power 0.5. Tăng power cũng làm đổi trọng số Prompt Injection,
nên không gộp thay đổi loss vào thử nghiệm hiệu chỉnh Portuguese này.

`WOLF_LOADING_MODE="v3_legacy"` tái lập cách đọc config/tokenizer của V3 bằng thư viện
đã khóa; đây không phải bảo đảm tái lập đúng từng bit hoặc cùng score.
`"checkpoint_config"` là cách chuẩn hóa RoPE ở V4, dành cho thử nghiệm riêng.
`ENABLE_PT_PAIR_TUNING=False` tắt hiệu chỉnh mới để chạy đối chứng cùng cấu hình.
`USE_FULL_TRAIN=False` mặc định: xuất selection trước, không tự train thêm 14.000 dòng.


In [ ]:
# Chon MOT backbone. Chay lai tu dau khi doi preset, khong tai dung logits cu.
MODEL_PRESET = "wolf"                       # "wolf" hoac "distilbert"; chi MOT backbone
WOLF_APPROVAL_CONFIRMED = True              # Challenge page 877551, 22/10/2026
WOLF_APPROVAL_SOURCE = "https://www.kaggle.com/competitions/rmit-hackathon-2026/overview/challenge"
WOLF_KAGGLE_HANDLE = "aisuko/wolf-defender-prompt-injection/transformers/default/1"
WOLF_LOCAL_PATH = ""                        # Tuy chon: folder model da tai/gan tren Kaggle

DATA_MODE = "upload"                       # "upload" hoac "path"
DATA_DIR = "/content/rmit-hackathon-2026"
MOUNT_DRIVE = False
OUTPUT_ROOT = "/content/rmit_c1_runs"        # Doi sang Drive de giu checkpoints
SEED = 42
SPLIT_SEED = 42
VAL_SIZE = 0.10
USE_FULL_TRAIN = False                     # Xem selection truoc; refit la tuy chon
FULL_TRAIN_EPOCHS = None                   # None = epoch cua checkpoint tot nhat
TRAIN_BATCH_SIZE = 8                       # OOM: giam 4, tang GRAD_ACCUM_STEPS len 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
USE_CLASS_WEIGHTS = True
GRADIENT_CHECKPOINTING = True

if MODEL_PRESET == "distilbert":
    MODEL_SOURCE = "distilbert/distilbert-base-multilingual-cased"
    MODEL_REVISION = "45c032ab32cc946ad88a166f7cb282f58c753c2e"
    EPOCHS, LEARNING_RATE, EARLY_STOPPING_PATIENCE = 8, 5e-5, 3
    MAX_LENGTH, LONG_TEXT_MODE = 512, "head_tail"
    CLASS_WEIGHT_POWER, FOCAL_GAMMA = 0.65, 2.0
elif MODEL_PRESET == "wolf":
    if not WOLF_APPROVAL_CONFIRMED or not WOLF_APPROVAL_SOURCE.strip():
        raise ValueError("Can doi chieu whitelist BTC cho Wolf va ghi WOLF_APPROVAL_SOURCE.")
    MODEL_SOURCE = WOLF_LOCAL_PATH          # Tu tai dung model version 1 neu de trong
    MODEL_REVISION = None
    EPOCHS, LEARNING_RATE, EARLY_STOPPING_PATIENCE = 4, 3e-5, 2
    MAX_LENGTH, LONG_TEXT_MODE = 256, "head"
    CLASS_WEIGHT_POWER, FOCAL_GAMMA = 0.5, 0.0  # Weighted CE de bat dau gan Melbourne
else:
    raise ValueError("MODEL_PRESET phai la distilbert hoac wolf.")
NOTEBOOK_VERSION = "v5_v3baseline_pt"
WOLF_LOADING_MODE = "v3_legacy"            # "checkpoint_config" la thu nghiem rieng
RUN_NAME = f"{MODEL_PRESET}_{NOTEBOOK_VERSION}"

# Cac tham so LoRA chi dung khi MODEL_PRESET == "distilbert".
LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 128, 0.05
LORA_TARGETS = ["q_lin", "k_lin", "v_lin", "out_lin", "lin1", "lin2"]
HEAD_MODULES = ["pre_classifier", "classifier"]
LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
LABEL2ID = {label: index for index, label in enumerate(LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}
LANGUAGES = ["pt", "fr", "it", "en"]
ENABLE_LEXICAL_BRANCH = True
LEXICAL_MAX_FEATURES, LEXICAL_C, LEXICAL_GLOBAL_WEIGHT = 300000, 2.0, 0.5
MIN_POSTPROCESS_GAIN = 0.002
ENABLE_PT_PAIR_TUNING = True
PT_PAIR_BIAS_VALUES = (-0.30, -0.15, 0.0, 0.15, 0.30)
PT_PAIR_MAX_MARGIN = 0.75
PT_PAIR_MIN_GAIN = 0.002
# Giu cac moc bias nho V2 va them range rong Melbourne; KHONG copy bias da fit cua ho.
POSTPROCESS_FIT_KWARGS = {
    "jailbreak_bias_values": [-1., -.75, -.5, -.3, -.15, 0., .15, .3, .5, .75, 1.],
    "injection_bias_values": [-1., -.75, -.5, -.3, -.15, 0., .15, .3, .5, .75, 1., 1.5, 2.],
    "bias_l2": 0.002,
}
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
assert DATA_MODE in {"upload", "path"}
assert LONG_TEXT_MODE in {"head", "head_tail"}
assert 0 < VAL_SIZE < 1
assert 8 <= MAX_LENGTH <= 512 and MAX_LENGTH % 8 == 0
assert TRAIN_BATCH_SIZE > 0 and GRAD_ACCUM_STEPS > 0
assert EPOCHS >= 1 and 0 <= CLASS_WEIGHT_POWER <= 1 and FOCAL_GAMMA >= 0
assert 0 <= LEXICAL_GLOBAL_WEIGHT <= 1
print("Notebook:", NOTEBOOK_VERSION, "| preset:", MODEL_PRESET, "| full refit:", USE_FULL_TRAIN)
print("Loading:", WOLF_LOADING_MODE, "| input:", MAX_LENGTH, LONG_TEXT_MODE)
print("Effective batch:", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)


In [ ]:
import gc
import io
import json
import hashlib
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import bitsandbytes as bnb
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from transformers import (
    AutoConfig, AutoTokenizer, AutoModelForSequenceClassification,
    BitsAndBytesConfig, DataCollatorWithPadding, Trainer,
    TrainingArguments, EarlyStoppingCallback, set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from IPython.display import display, FileLink

if not torch.cuda.is_available():
    raise RuntimeError("Can GPU CUDA: Runtime > Change runtime type > GPU, roi chay lai.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("Notebook yeu cau mot GPU. Restart runtime de CUDA_VISIBLE_DEVICES co hieu luc.")

set_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
# T4 (capability 7.5) khong co native BF16, du PyTorch co the cho phep emulation.
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
USE_BF16 = GPU_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
PACKAGE_VERSIONS = {
    name: version(name)
    for name in ["torch", "transformers", "peft", "accelerate", "bitsandbytes",
                 "huggingface-hub", "numpy", "pandas", "scikit-learn", "safetensors", "joblib", "kagglehub"]
}
for package, wanted in PINS.items():
    if PACKAGE_VERSIONS[package] != wanted:
        raise RuntimeError(f"Sai version {package}; restart runtime va chay lai cell cai dat.")

DATA_PATH = Path(DATA_DIR)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = Path(OUTPUT_ROOT) / f"{RUN_NAME}_{RUN_ID}"
ARTIFACT_DIR = RUN_DIR / "artifacts"
ADAPTER_DIR = ARTIFACT_DIR / ("selection_model" if MODEL_PRESET == "wolf" else "selection_adapter")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)

print("Python:", sys.version)
print("GPU capability:", GPU_CAPABILITY)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Compute dtype:", COMPUTE_DTYPE)
print("Output:", RUN_DIR)
print("Training preset:", MODEL_PRESET)
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["package", "version"]))


## 3. Upload dữ liệu

Chọn cả ba CSV trong một lần hoặc upload ZIP từ Kaggle. Nếu dùng ZIP, notebook chỉ lấy ba CSV có tên đúng, kể cả khi chúng nằm trong thư mục con. Không cần Kaggle token và không tải dữ liệu từ GitHub.


In [ ]:
EXPECTED_FILES = {"train.csv", "test.csv", "sample_submission.csv"}

if DATA_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    selected = {}

    def add_csv(name, payload):
        if name in selected:
            raise ValueError(f"Trung file {name} trong lan upload. Chi chon mot ban moi file.")
        selected[name] = payload

    for upload_name, payload in uploaded.items():
        filename = Path(upload_name).name
        if filename in EXPECTED_FILES:
            add_csv(filename, payload)
        elif filename.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(payload)) as archive:
                for member in archive.infolist():
                    name = Path(member.filename).name
                    if not member.is_dir() and name in EXPECTED_FILES:
                        add_csv(name, archive.read(member))

    missing = EXPECTED_FILES - set(selected)
    if missing:
        raise ValueError(f"Upload thieu {sorted(missing)}. Chay lai cell va chon ca ba CSV/ZIP.")
    DATA_PATH.mkdir(parents=True, exist_ok=True)
    for name, payload in selected.items():
        (DATA_PATH / name).write_bytes(payload)
    del uploaded, selected

missing = [name for name in EXPECTED_FILES if not (DATA_PATH / name).is_file()]
if missing:
    raise FileNotFoundError(f"Khong tim thay {missing} trong {DATA_PATH}. Kiem tra DATA_DIR.")
print("Data directory:", DATA_PATH)


In [ ]:
def load_csv(name):
    # Giu nguyen text va ID, khong bien chuoi "NA" thanh gia tri thieu.
    return pd.read_csv(DATA_PATH / name, dtype=str, keep_default_na=False, encoding="utf-8-sig")

train_df = load_csv("train.csv")
test_df = load_csv("test.csv")
sample_df = load_csv("sample_submission.csv")

def validate_frame(frame, name, columns, expected_rows):
    if list(frame.columns) != columns:
        raise ValueError(f"{name}: can cot {columns}; nhan duoc {list(frame.columns)}.")
    if len(frame) != expected_rows:
        raise ValueError(f"{name}: can {expected_rows} dong; nhan duoc {len(frame)}.")
    if frame["id"].isna().any() or frame["id"].str.strip().eq("").any() or not frame["id"].is_unique:
        raise ValueError(f"{name}: ID rong hoac trung.")
    for column in ["text", "language"]:
        if column in frame and (
            frame[column].isna().any() or frame[column].str.strip().eq("").any()
        ):
            raise ValueError(f"{name}: {column} rong.")
    if "language" in frame and not set(frame["language"]).issubset(LANGUAGES):
        raise ValueError(f"{name}: language ngoai mien {LANGUAGES}.")

validate_frame(train_df, "train.csv", ["id", "text", "language", "label"], 14000)
validate_frame(test_df, "test.csv", ["id", "text", "language"], 6000)
validate_frame(sample_df, "sample_submission.csv", ["id", "label"], 6000)
if set(train_df["label"]) != set(LABELS):
    raise ValueError("Train phai chua du ba nhan chinh xac.")
if set(sample_df["id"]) != set(test_df["id"]):
    raise ValueError("ID sample_submission khong khop test.")
if train_df["text"].duplicated().any():
    raise ValueError("Train co text trung; can group-aware split de tranh leakage.")

train_df["label_id"] = train_df["label"].map(LABEL2ID).astype("int64")
DATA_HASHES = {
    name: hashlib.sha256((DATA_PATH / name).read_bytes()).hexdigest()
    for name in sorted(EXPECTED_FILES)
}
display(pd.crosstab(train_df["language"], train_df["label"]).reindex(columns=LABELS))
display(train_df["text"].str.len().describe().rename("text_characters").to_frame())
print("Train:", len(train_df), "| Test:", len(test_df))
print("Sample labels chi la placeholder; khong dung lam ground truth.")


## 4. Holdout để chọn checkpoint và class weights

Luôn chia 90/10 theo language × label, kể cả khi `USE_FULL_TRAIN=True`.
Chỉ bước refit cuối mới dùng toàn bộ train; không dùng validation đã học để chọn model.
Class weights được tính riêng trên tập train của từng bước.


In [ ]:
strata = train_df["language"] + "::" + train_df["label"]
smallest_group = int(strata.value_counts().min())
if smallest_group < 2:
    raise ValueError("Nhom language/label qua nho; can xem lai cach split.")

fit_df, valid_df = train_test_split(
    train_df, test_size=VAL_SIZE, random_state=SPLIT_SEED, stratify=strata,
)
fit_df = fit_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
assert set(fit_df["id"]).isdisjoint(set(valid_df["id"]))
assert set(fit_df["text"]).isdisjoint(set(valid_df["text"]))
assert set(fit_df["id"]) | set(valid_df["id"]) == set(train_df["id"])

split_manifest = pd.concat([
    fit_df[["id", "language", "label"]].assign(split="train"),
    valid_df[["id", "language", "label"]].assign(split="validation"),
], ignore_index=True)
split_manifest.to_csv(ARTIFACT_DIR / "split.csv", index=False)

def compute_class_weights(label_ids, num_classes, power=0.65, enabled=True):
    ids = np.asarray(label_ids, dtype=np.int64)
    if ids.ndim != 1 or ids.size == 0 or not np.isin(ids, np.arange(num_classes)).all():
        raise ValueError("Label IDs khong hop le.")
    if not 0 <= power <= 1:
        raise ValueError("Class weight power phai trong [0, 1].")
    counts = np.bincount(ids, minlength=num_classes)
    if (counts == 0).any():
        raise ValueError("Moi class phai co mau train.")
    values = (len(ids) / counts.astype(np.float64)) ** (power if enabled else 0.0)
    values /= np.average(values, weights=counts)
    return values.astype(np.float32)

class_counts = np.bincount(fit_df["label_id"].to_numpy(), minlength=len(LABELS))
power = CLASS_WEIGHT_POWER if USE_CLASS_WEIGHTS else 0.0
weight_values = compute_class_weights(fit_df["label_id"], len(LABELS), power, USE_CLASS_WEIGHTS)
class_weights = torch.tensor(weight_values, dtype=torch.float32)
print("Selection training:", len(fit_df), "| Validation:", len(valid_df))
display(pd.DataFrame({"label": LABELS, "train_count": class_counts, "weight": weight_values}))


### Nhánh bổ trợ TF-IDF ký tự + SVM

Học prompt không cắt ngắn, gồm classifier chung và classifier theo ngôn ngữ.
Ở selection, chỉ fit trên 12.600 mẫu; validation/test chỉ transform/predict.
Ở full refit, fit lại trên 14.000 mẫu có nhãn. Không fit trên test.csv.
Global/local cố định 50/50; cross-fit chọn mức phối hợp với backbone.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
import joblib

def fit_lexical_branch(frame):
    # Chi fit tren frame co nhan cua tung stage; khong fit tren test.
    vectorizer = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 6), min_df=2,
        max_features=LEXICAL_MAX_FEATURES, sublinear_tf=True, dtype=np.float32,
    )
    features = vectorizer.fit_transform(frame["text"].tolist())
    y = frame["label_id"].to_numpy()
    languages = frame["language"].to_numpy()

    def fit_classifier(x, labels):
        classifier = LinearSVC(
            C=LEXICAL_C, class_weight="balanced", dual="auto",
            max_iter=5000, random_state=SEED,
        )
        classifier.fit(x, labels)
        assert np.array_equal(classifier.classes_, np.arange(len(LABELS)))
        return classifier

    global_model = fit_classifier(features, y)
    language_models = {}
    for language in LANGUAGES:
        mask = languages == language
        if set(y[mask]) == set(range(len(LABELS))):
            language_models[language] = fit_classifier(features[mask], y[mask])
    return {"vectorizer": vectorizer, "global_model": global_model,
            "language_models": language_models, "labels": list(LABELS),
            "global_weight": LEXICAL_GLOBAL_WEIGHT}

def lexical_predict(bundle, frame):
    assert bundle["labels"] == LABELS, "Sai thu tu nhan lexical model."
    features = bundle["vectorizer"].transform(frame["text"].tolist())
    global_scores = bundle["global_model"].decision_function(features)
    local_scores = global_scores.copy()
    for language, classifier in bundle["language_models"].items():
        mask = frame["language"].eq(language).to_numpy()
        if mask.any():
            local_scores[mask] = classifier.decision_function(features[mask])
    weight = bundle["global_weight"]
    scores = weight * global_scores + (1.0 - weight) * local_scores
    assert scores.shape == (len(frame), len(LABELS)) and np.isfinite(scores).all()
    return scores.astype(np.float32)

valid_lexical_scores = test_lexical_scores = None
lexical_bundle = None
if ENABLE_LEXICAL_BRANCH:
    lexical_bundle = fit_lexical_branch(fit_df)
    valid_lexical_scores = lexical_predict(lexical_bundle, valid_df)
    test_lexical_scores = lexical_predict(lexical_bundle, test_df)
    lexical_f1 = f1_score(
        valid_df["label_id"], valid_lexical_scores.argmax(axis=1),
        labels=np.arange(len(LABELS)), average="macro", zero_division=0,
    )
    print("Lexical validation Macro F1:", round(lexical_f1, 6))
    print("Features:", len(lexical_bundle["vectorizer"].vocabulary_))
    # Can bundle nay + adapter + postprocess.json de tai lap ensemble inference.
    joblib.dump(lexical_bundle, ARTIFACT_DIR / "selection_lexical.joblib", compress=3)
    np.save(ARTIFACT_DIR / "validation_lexical_scores.npy", valid_lexical_scores)
    np.save(ARTIFACT_DIR / "test_lexical_scores.npy", test_lexical_scores)
else:
    print("Lexical branch disabled; submission se dung backbone da chon.")


## 5. Nạp model theo cấu hình đã chọn

Mặc định `v3_legacy` giữ cách AutoConfig 4.57.6 đọc config nguồn như V3 và cùng
cách sửa metadata tokenizer của V3. Chỉ sao chép các file config/tokenizer nhỏ,
không sao chép weights. Các giá trị RoPE thực sự được ghi vào `model_provenance.json`.
`checkpoint_config` giữ cách ánh xạ metadata mới ở V4 cho thử nghiệm riêng.

256 token `head` là baseline quan sát; báo cáo `token_lengths.csv` vẫn lưu phần
bị cắt theo ngôn ngữ/lớp. Chưa kết luận context dài hơn luôn tốt hoặc xấu hơn.


In [ ]:
"""Standalone cell: preserve V3 loading semantics or opt into checkpoint config.

The V3 profile reproduces its loader choices, not its measured score or bitwise
GPU results. Both profiles read the approved source without editing its files.
"""


def normalize_wolf_config(raw_config):
    """Translate supported v5 ModernBERT settings to Transformers 4.57.6."""
    from copy import deepcopy
    import math

    config = deepcopy(raw_config)
    if config.get("model_type") != "modernbert":
        raise ValueError("Wolf source must contain a ModernBERT config.")
    changes = {}
    rope = config.get("rope_parameters")
    if rope is not None:
        if not isinstance(rope, dict) or set(rope) != {"full_attention", "sliding_attention"}:
            raise ValueError("Unsupported Wolf rope_parameters: expected full/sliding attention settings.")
        for kind, legacy_name in (("full_attention", "global_rope_theta"),
                                  ("sliding_attention", "local_rope_theta")):
            settings = rope[kind]
            if not isinstance(settings, dict) or settings.get("rope_type", "default") != "default":
                raise ValueError(f"Unsupported {kind} RoPE type; do not silently approximate this checkpoint.")
            if set(settings) - {"rope_type", "rope_theta"}:
                raise ValueError(f"Unsupported extra {kind} RoPE settings: {settings}.")
            theta = settings.get("rope_theta")
            if isinstance(theta, bool) or not isinstance(theta, (int, float)):
                raise ValueError(f"Missing/nonnumeric {kind} rope_theta.")
            if not math.isfinite(theta) or theta <= 0:
                raise ValueError(f"Invalid {kind} rope_theta.")
            if legacy_name in config and config[legacy_name] is not None:
                if float(config[legacy_name]) != float(theta):
                    raise ValueError(f"Conflicting {legacy_name} and rope_parameters in source config.")
            config[legacy_name] = float(theta)
            changes[legacy_name] = float(theta)
        # The older implementation consumes the translated scalar fields.
        config.pop("rope_parameters")

    for name in ("global_rope_theta", "local_rope_theta"):
        if name in config and config[name] is not None:
            value = config[name]
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise ValueError(f"Invalid {name} in source config.")
            if not math.isfinite(value) or value <= 0:
                raise ValueError(f"Invalid {name} in source config.")
    if config.get("rope_scaling") is not None:
        raise ValueError("Scaled RoPE needs a separate compatibility implementation.")

    layer_types = config.get("layer_types")
    if layer_types is not None:
        depth = config.get("num_hidden_layers")
        interval = config.get("global_attn_every_n_layers", 3)
        if type(depth) is not int or depth <= 0 or type(interval) is not int or interval <= 0:
            raise ValueError("Invalid ModernBERT layer count/attention interval.")
        expected = ["full_attention" if index % interval == 0 else "sliding_attention"
                    for index in range(depth)]
        if layer_types != expected:
            raise ValueError("Source layer_types cannot be reproduced by this pinned ModernBERT runtime.")
        config.pop("layer_types")
        changes["layer_schedule_verified"] = True

    if "norm_eps" not in config and "layer_norm_eps" in config:
        config["norm_eps"] = config["layer_norm_eps"]
        changes["norm_eps"] = config["norm_eps"]
    # Keep checkpoint pooling (Wolf uses mean); do not replace it with CLS.
    if config.get("classifier_pooling", "cls") not in {"mean", "cls"}:
        raise ValueError("Unsupported classifier pooling in Wolf source.")
    return config, {
        "source_transformers_version": raw_config.get("transformers_version"),
        "source_rope_parameters": deepcopy(rope),
        "changes": changes,
        "classifier_pooling": config.get("classifier_pooling", "cls"),
    }


def load_wolf_tokenizer(source_dir, compatibility_dir):
    """Load the real fast-tokenizer graph, preserving its IDs and postprocessor."""
    import hashlib
    import json
    from pathlib import Path
    from tokenizers import AddedToken, Tokenizer
    from transformers import PreTrainedTokenizerFast

    source_dir, compatibility_dir = Path(source_dir), Path(compatibility_dir)
    tokenizer_file = source_dir / "tokenizer.json"
    config_file = source_dir / "tokenizer_config.json"
    if not tokenizer_file.is_file() or not config_file.is_file():
        raise ValueError("Wolf folder must include tokenizer.json and tokenizer_config.json.")
    settings = json.loads(config_file.read_text(encoding="utf-8"))
    # Construct explicitly instead of dispatching the v5 TokenizersBackend name
    # through the v4 AutoTokenizer registry. No model weights are copied.
    backend = Tokenizer.from_file(str(tokenizer_file))
    original_vocab = backend.get_vocab(with_added_tokens=True)

    def token_value(value):
        if isinstance(value, str):
            content, normalized = value, value
        elif isinstance(value, dict) and isinstance(value.get("content"), str):
            content = value["content"]
            fields = {key: item for key, item in value.items()
                      if key in {"content", "single_word", "lstrip", "rstrip", "normalized", "special"}}
            normalized = AddedToken(**fields)
        else:
            raise ValueError(f"Unsupported special-token entry: {value!r}")
        if content not in original_vocab:
            raise ValueError(f"Special token {content!r} is absent from tokenizer.json; refusing to add vocabulary.")
        return normalized

    arguments = {
        key: settings[key] for key in (
            "model_max_length", "padding_side", "truncation_side",
            "clean_up_tokenization_spaces", "spaces_between_special_tokens",
        ) if key in settings
    }
    for key in ("bos_token", "eos_token", "unk_token", "sep_token", "pad_token", "cls_token", "mask_token"):
        if settings.get(key) is not None:
            arguments[key] = token_value(settings[key])
    extra = settings.get("extra_special_tokens", [])
    if isinstance(extra, dict):
        extra = list(extra.values())
    if not isinstance(extra, list):
        raise ValueError("extra_special_tokens must be a list or mapping.")
    additional = settings.get("additional_special_tokens", [])
    if not isinstance(additional, list):
        raise ValueError("additional_special_tokens must be a list.")
    seen, normalized_extra = set(), []
    for value in additional + extra:
        normalized = token_value(value)
        content = str(normalized)
        if content not in seen:
            seen.add(content)
            normalized_extra.append(normalized)
    if normalized_extra:
        arguments["additional_special_tokens"] = normalized_extra
    tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend, **arguments)
    tokenizer.model_input_names = ["input_ids", "attention_mask"]
    if tokenizer.get_vocab() != original_vocab:
        raise ValueError("Tokenizer compatibility changed token IDs or vocabulary.")
    if tokenizer.pad_token_id is None:
        raise ValueError("Wolf tokenizer needs a padding token for dynamic batching.")
    # Only tokenizer/config assets are written under this run's own directory.
    compatibility_dir.mkdir(parents=True, exist_ok=True)
    tokenizer.save_pretrained(str(compatibility_dir))
    report = {
        "source_tokenizer_class": settings.get("tokenizer_class"),
        "runtime_tokenizer_class": type(tokenizer).__name__,
        "normalized_extra_special_tokens": [str(value) for value in normalized_extra],
        "tokenizer_json_sha256": hashlib.sha256(tokenizer_file.read_bytes()).hexdigest(),
        "tokenizer_config_sha256": hashlib.sha256(config_file.read_bytes()).hexdigest(),
        "vocabulary_unchanged": True,
        "postprocessor_preserved": True,
        "compatibility_directory": str(compatibility_dir),
    }
    return tokenizer, report


def load_v3_wolf_tokenizer(source_dir, compatibility_dir):
    """Apply the executed V3 workaround to tokenizer assets, never model weights."""
    import hashlib
    import json
    import shutil
    from pathlib import Path
    from transformers import AutoTokenizer

    source_dir, compatibility_dir = Path(source_dir), Path(compatibility_dir)
    asset_names = (
        "config.json", "tokenizer.json", "tokenizer_config.json",
        "special_tokens_map.json", "added_tokens.json", "vocab.json",
        "vocab.txt", "merges.txt", "tokenizer.model", "spiece.model",
        "sentencepiece.bpe.model",
    )
    required = asset_names[:3]
    if any(not (source_dir / name).is_file() for name in required):
        raise ValueError("Wolf folder must include config.json, tokenizer.json and tokenizer_config.json.")
    if compatibility_dir.resolve().is_relative_to(source_dir.resolve()):
        raise ValueError("Tokenizer compatibility folder must be outside the original source.")
    compatibility_dir.mkdir(parents=True, exist_ok=True)
    copied, hashes = [], {}
    for name in asset_names:
        source = source_dir / name
        if source.is_file():
            # Copy content only, so read-only source permission bits do not
            # prevent the compatibility copy's tokenizer metadata edit.
            shutil.copyfile(source, compatibility_dir / name)
            copied.append(name)
            hashes[name] = hashlib.sha256(source.read_bytes()).hexdigest()
    staged_config = compatibility_dir / "tokenizer_config.json"
    settings = json.loads(staged_config.read_text(encoding="utf-8"))
    original_tokenizer_class = settings.get("tokenizer_class")
    removed = []
    # These are the two edits made in the archived executed V3 notebook.
    if settings.get("tokenizer_class") == "TokenizersBackend":
        settings.pop("tokenizer_class")
        removed.append("tokenizer_class")
    if isinstance(settings.get("extra_special_tokens"), list):
        settings.pop("extra_special_tokens")
        removed.append("extra_special_tokens")
    staged_config.write_text(json.dumps(settings, indent=2, ensure_ascii=False), encoding="utf-8")
    tokenizer = AutoTokenizer.from_pretrained(
        str(compatibility_dir), use_fast=False, trust_remote_code=False,
    )
    # ModernBERT has no slow tokenizer in Transformers 4.57.6, so this V3
    # AutoTokenizer call still resolves to PreTrainedTokenizerFast.
    report = {
        "loader": "AutoTokenizer.from_pretrained",
        "use_fast_argument": False,
        "source_tokenizer_class": original_tokenizer_class,
        "runtime_tokenizer_class": type(tokenizer).__name__,
        "removed_v3_incompatible_fields": removed,
        "copied_assets": copied,
        "source_asset_sha256": hashes,
        "tokenizer_json_sha256": hashes["tokenizer.json"],
        "tokenizer_config_sha256": hashes["tokenizer_config.json"],
        "compatibility_directory": str(compatibility_dir),
        "model_weights_copied": False,
    }
    return tokenizer, report


def load_wolf_assets(source_dir, compatibility_dir, loading_mode):
    """Keep V3's raw AutoConfig defaults unless the other profile is selected."""
    import hashlib
    import json
    from pathlib import Path
    from transformers import AutoConfig

    if loading_mode not in {"v3_legacy", "checkpoint_config"}:
        raise ValueError("WOLF_LOADING_MODE must be v3_legacy or checkpoint_config.")
    source_dir, compatibility_dir = Path(source_dir), Path(compatibility_dir)
    source_config_file = source_dir / "config.json"
    source_config = json.loads(source_config_file.read_text(encoding="utf-8"))
    if source_config.get("model_type") != "modernbert":
        raise ValueError("Wolf source must contain a ModernBERT config.")
    # Separate staged folders prevent a profile switch from reusing files saved
    # by the other loader in an interactive runtime.
    staged_dir = compatibility_dir / loading_mode
    if loading_mode == "v3_legacy":
        config = AutoConfig.from_pretrained(str(source_dir), trust_remote_code=False)
        tokenizer, tokenizer_report = load_v3_wolf_tokenizer(source_dir, staged_dir)
        config_report = {
            "loader": "AutoConfig.from_pretrained",
            "source_transformers_version": source_config.get("transformers_version"),
            "source_rope_parameters": source_config.get("rope_parameters"),
            "changes": {},
            "classifier_pooling": config.classifier_pooling,
            "note": (
                "Preserves the executed V3 raw-config loading behavior in Transformers 4.57.6. "
                "Transformers 5 rope_parameters are not translated; runtime legacy defaults remain active. "
                "This restores loader choices, not a guarantee of identical training results."
            ),
        }
    else:
        normalized_config, config_report = normalize_wolf_config(source_config)
        config = AutoConfig.for_model(normalized_config.pop("model_type"), **normalized_config)
        config_report["loader"] = "AutoConfig.for_model with normalized checkpoint settings"
        tokenizer, tokenizer_report = load_wolf_tokenizer(source_dir, staged_dir)
    report = {
        "loading_mode": loading_mode,
        "config": config_report,
        "tokenizer": tokenizer_report,
        "source_config_sha256": hashlib.sha256(source_config_file.read_bytes()).hexdigest(),
        "resolved_global_rope_theta": config.global_rope_theta,
        "resolved_local_rope_theta": config.local_rope_theta,
    }
    return config, tokenizer, report


MODEL_COMPATIBILITY = {"preset": MODEL_PRESET}
if MODEL_PRESET == "wolf":
    # Reset only Wolf's temporary runtime source; keep DistilBERT's pinned ID.
    MODEL_SOURCE = WOLF_LOCAL_PATH
    if not MODEL_SOURCE:
        import kagglehub
        MODEL_SOURCE = kagglehub.model_download(WOLF_KAGGLE_HANDLE)
    model_root = Path(MODEL_SOURCE)
    candidates = ([model_root / "config.json"] if (model_root / "config.json").is_file()
                  else list(model_root.rglob("config.json")))
    if len(candidates) != 1:
        raise ValueError("WOLF_LOCAL_PATH must identify exactly one model folder with config.json.")
    MODEL_SOURCE = str(candidates[0].parent)
    base_config, tokenizer, compatibility_report = load_wolf_assets(
        MODEL_SOURCE, RUN_DIR / "wolf_tokenizer_compat", WOLF_LOADING_MODE,
    )
    MODEL_COMPATIBILITY.update(compatibility_report)
else:
    base_config = AutoConfig.from_pretrained(
        MODEL_SOURCE, revision=MODEL_REVISION, trust_remote_code=False,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_SOURCE, revision=getattr(base_config, "_commit_hash", None) or MODEL_REVISION,
        use_fast=True, trust_remote_code=False,
    )

expected_type = "modernbert" if MODEL_PRESET == "wolf" else "distilbert"
if base_config.model_type != expected_type:
    raise ValueError(f"Preset {MODEL_PRESET} requires model_type={expected_type}.")
if MAX_LENGTH > base_config.max_position_embeddings:
    raise ValueError("MAX_LENGTH exceeds checkpoint positional capacity.")
if max(tokenizer.get_vocab().values()) >= base_config.vocab_size:
    raise ValueError("Tokenizer IDs exceed checkpoint vocabulary.")
if tokenizer.pad_token_id != base_config.pad_token_id:
    raise ValueError("Tokenizer/checkpoint padding IDs differ.")
RESOLVED_MODEL_REVISION = getattr(base_config, "_commit_hash", None)
LOAD_KWARGS = {"trust_remote_code": False}
if MODEL_PRESET == "distilbert":
    LOAD_KWARGS["revision"] = RESOLVED_MODEL_REVISION or MODEL_REVISION
MODEL_PROVENANCE = {
    "preset": MODEL_PRESET, "source": MODEL_SOURCE,
    "revision": LOAD_KWARGS.get("revision"),
    "kaggle_handle": WOLF_KAGGLE_HANDLE if MODEL_PRESET == "wolf" else None,
    "approval_source": WOLF_APPROVAL_SOURCE if MODEL_PRESET == "wolf" else "docs/requirements.md",
    "compatibility": MODEL_COMPATIBILITY,
}
(ARTIFACT_DIR / "model_compatibility.json").write_text(
    json.dumps(MODEL_COMPATIBILITY, indent=2, ensure_ascii=False), encoding="utf-8",
)
(ARTIFACT_DIR / "model_provenance.json").write_text(
    json.dumps(MODEL_PROVENANCE, indent=2, ensure_ascii=False), encoding="utf-8",
)


def encode_texts(texts):
    # Preserve the actual tokenizer postprocessor, including inserted tokens.
    encoded = tokenizer(
        texts, add_special_tokens=True, truncation=False,
        return_special_tokens_mask=True, return_attention_mask=False,
        return_token_type_ids=False,
    )
    final_ids, lengths = [], []
    for token_ids, special_mask in zip(encoded["input_ids"], encoded["special_tokens_mask"]):
        positions = [index for index, special in enumerate(special_mask) if not special]
        lengths.append(len(positions))
        budget = MAX_LENGTH - sum(special_mask)
        if budget < 1:
            raise ValueError("MAX_LENGTH cannot fit special tokens and content.")
        if len(positions) > budget:
            if LONG_TEXT_MODE == "head":
                chosen = positions[:budget]
            else:
                head_count = (budget + 1) // 2
                tail_count = budget - head_count
                chosen = positions[:head_count] + (positions[-tail_count:] if tail_count else [])
            kept = set(chosen)
            token_ids = [token_id for index, token_id in enumerate(token_ids)
                         if special_mask[index] or index in kept]
        assert len(token_ids) <= MAX_LENGTH
        final_ids.append(token_ids)
    return {"input_ids": final_ids, "attention_mask": [[1] * len(ids) for ids in final_ids]}, lengths


class PromptDataset(Dataset):
    def __init__(self, frame, labelled):
        self.features = {"input_ids": [], "attention_mask": []}
        self.labels = frame["label_id"].tolist() if labelled else None
        self.raw_lengths = []
        for start in range(0, len(frame), 256):
            encoded, lengths = encode_texts(frame["text"].iloc[start:start + 256].tolist())
            self.raw_lengths.extend(lengths)
            for key in self.features:
                self.features[key].extend(encoded[key])

    def __len__(self):
        return len(self.features["input_ids"])

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.features.items()}
        if self.labels is not None:
            item["labels"] = int(self.labels[index])
        return item


train_dataset = PromptDataset(fit_df, labelled=True)
valid_dataset = PromptDataset(valid_df, labelled=True)
test_dataset = PromptDataset(test_df, labelled=False)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
budget = MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False)
length_reports = []
for split, frame, dataset in [("train", fit_df, train_dataset),
                              ("validation", valid_df, valid_dataset),
                              ("test", test_df, test_dataset)]:
    report = frame[["id", "language"]].copy()
    report["label"] = frame["label"].to_numpy() if "label" in frame else "UNKNOWN"
    report["split"] = split
    report["raw_tokens"] = dataset.raw_lengths
    report["truncated"] = report["raw_tokens"] > budget
    length_reports.append(report)
token_lengths = pd.concat(length_reports, ignore_index=True)
token_lengths.to_csv(ARTIFACT_DIR / "token_lengths.csv", index=False)
display(token_lengths.groupby(["split", "language", "label"])["truncated"].agg(["size", "sum", "mean"]))
print("Datasets:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("Model revision:", RESOLVED_MODEL_REVISION or MODEL_REVISION)
print("Model compatibility:", json.dumps(MODEL_COMPATIBILITY, indent=2, ensure_ascii=False))


## 6. Khởi tạo backbone đã chọn

Wolf dùng full fine-tune, head 3 lớp được khởi tạo lại, SDPA và AMP; tắt compile nội bộ
ModernBERT để chạy đơn giản trên T4. DistilBERT dùng QLoRA NF4 và train cả hai lớp head.
Factory luôn khởi tạo mới từ pretrained cho cả selection và full refit.


In [ ]:
def build_model():
    """Reload the approved source and reset the seed for selection or final refit."""
    from copy import deepcopy

    set_seed(SEED)
    if MODEL_PRESET == "wolf":
        if base_config.model_type != "modernbert":
            raise ValueError("Preset wolf yeu cau checkpoint Wolf-Defender ModernBERT.")
        config = deepcopy(base_config)
        config.num_labels = len(LABELS)
        config.id2label = dict(ID2LABEL)
        config.label2id = dict(LABEL2ID)
        config.problem_type = "single_label_classification"
        # ModernBERT co torch.compile noi bo, doc lap voi Trainer.torch_compile.
        # Dat lai False moi lan load: ModernBertConfig.to_dict() bo truong nay.
        config.reference_compile = False
        loaded = AutoModelForSequenceClassification.from_pretrained(
            MODEL_SOURCE,
            config=config,
            # Checkpoint Wolf co 2 nhan; khoi tao lai classifier cho 3 nhan C1.
            ignore_mismatched_sizes=True,
            dtype=torch.float32,
            attn_implementation="sdpa",
            use_safetensors=True,
            **LOAD_KWARGS,
        )
        assert loaded.config.model_type == "modernbert"
        assert loaded.config.reference_compile is False
        assert loaded.classifier.out_features == len(LABELS)
        assert not getattr(loaded, "is_loaded_in_4bit", False)
        # Full fine-tuning: FP32 parameters; Trainer bat AMP FP16/BF16.
        assert all(parameter.requires_grad for parameter in loaded.parameters())
        assert all(parameter.dtype == torch.float32 for parameter in loaded.parameters())
        print("Wolf-Defender: full fine-tuning, 3-label classifier, SDPA, reference_compile=False.")
    elif MODEL_PRESET == "distilbert":
        if base_config.model_type != "distilbert":
            raise ValueError("Preset distilbert yeu cau DistilBERT multilingual.")
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
            # Transformers 4.57.6 su dung skip_modules cho ca 4-bit quantizer.
            llm_int8_skip_modules=list(HEAD_MODULES),
        )
        loaded = AutoModelForSequenceClassification.from_pretrained(
            MODEL_SOURCE,
            num_labels=len(LABELS),
            id2label=ID2LABEL,
            label2id=LABEL2ID,
            problem_type="single_label_classification",
            quantization_config=quant_config,
            device_map={"": 0},
            dtype=COMPUTE_DTYPE,
            attn_implementation="sdpa",
            use_safetensors=True,
            **LOAD_KWARGS,
        )
        assert getattr(loaded, "is_loaded_in_4bit", False), "Backbone chua load 4-bit."
        assert any(isinstance(module, bnb.nn.Linear4bit) for module in loaded.modules())
        for name in HEAD_MODULES:
            assert not isinstance(getattr(loaded, name), bnb.nn.Linear4bit), f"{name} khong duoc quantize."
        loaded = prepare_model_for_kbit_training(
            loaded,
            use_gradient_checkpointing=GRADIENT_CHECKPOINTING,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )
        for suffix in LORA_TARGETS:
            hits = [name for name, module in loaded.named_modules() if name.endswith("." + suffix)]
            assert len(hits) == base_config.n_layers, (suffix, hits)
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGETS,
            modules_to_save=list(HEAD_MODULES),
            bias="none",
        )
        loaded = get_peft_model(loaded, lora_config)
        trainable_names = [name for name, parameter in loaded.named_parameters() if parameter.requires_grad]
        assert trainable_names
        assert any("lora_" in name for name in trainable_names)
        for head in HEAD_MODULES:
            assert any(f".{head}.modules_to_save." in name for name in trainable_names)
        assert all(
            "lora_" in name or any(f".{head}.modules_to_save." in name for head in HEAD_MODULES)
            for name in trainable_names
        ), "Co tham so backbone dang train ngoai adapter/head."
        loaded.print_trainable_parameters()
    else:
        raise ValueError(f"MODEL_PRESET khong hop le: {MODEL_PRESET}")

    assert loaded.config.num_labels == len(LABELS)
    assert loaded.config.id2label == ID2LABEL
    trainable_names = [name for name, parameter in loaded.named_parameters() if parameter.requires_grad]
    trainable_count = sum(parameter.numel() for parameter in loaded.parameters() if parameter.requires_grad)
    total_count = sum(parameter.numel() for parameter in loaded.parameters())
    print(f"Trainable: {trainable_count:,} / {total_count:,} ({100 * trainable_count / total_count:.2f}%)")
    (ARTIFACT_DIR / "trainable_parameters.txt").write_text(
        "\n".join(trainable_names) + "\n", encoding="utf-8",
    )
    return loaded


model = build_model()


## 7. Macro F1 và loss

Weighted focal loss tính `p_t` từ cross-entropy **chưa nhân class weight**, giữ gradient
qua hệ số focal. `gamma=0` trở về weighted CE. Mẫu số là tổng số mẫu trong bước
gradient accumulation, kể cả microbatch cuối nhỏ hơn. Wolf mặc định CE như Melbourne;
DistilBERT mặc định focal gamma 2 theo kế hoạch. Không suy ra gain trước khi chạy.


In [ ]:
def compute_metrics(prediction):
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted = np.argmax(logits, axis=-1)
    expected = prediction.label_ids
    class_f1 = f1_score(
        expected, predicted, labels=list(range(len(LABELS))),
        average=None, zero_division=0,
    )
    result = {
        "macro_f1": float(class_f1.mean()),
        "accuracy": float(accuracy_score(expected, predicted)),
    }
    result.update({f"f1_{label.lower()}": float(score) for label, score in zip(LABELS, class_f1)})
    return result

def make_weighted_focal_loss(weights, gamma):
    if gamma < 0:
        raise ValueError("Focal gamma phai >= 0.")
    # Bind weights rieng cho tung Trainer; full refit khong doi closure cua selection.
    fixed_weights = torch.as_tensor(weights, dtype=torch.float32).clone()

    def weighted_focal_loss(outputs, labels, num_items_in_batch=None):
        logits = outputs.logits.float()
        ce = F.cross_entropy(logits, labels, reduction="none")
        pt = torch.exp(-ce)
        alpha = fixed_weights.to(device=logits.device, dtype=logits.dtype)[labels]
        losses = alpha * (1.0 - pt).pow(gamma) * ce
        count = labels.numel() if num_items_in_batch is None else num_items_in_batch
        denominator = torch.as_tensor(count, device=logits.device, dtype=logits.dtype).clamp_min(1)
        return losses.sum() / denominator

    return weighted_focal_loss


def selected_checkpoint_epoch(state):
    # early stopping co the ket thuc muon hon checkpoint tot nhat.
    import math
    checkpoint = getattr(state, "best_model_checkpoint", None)
    if not checkpoint:
        raise ValueError("Thieu best_model_checkpoint de chon epoch refit.")
    try:
        step = int(str(checkpoint).replace("\\", "/").rstrip("/").rsplit("checkpoint-", 1)[1])
    except (ValueError, IndexError) as exc:
        raise ValueError("Khong doc duoc step cua best checkpoint.") from exc
    matches = [row for row in state.log_history
               if row.get("step") == step and "eval_macro_f1" in row and "epoch" in row]
    if not matches:
        raise ValueError("Khong co eval log khop best checkpoint.")
    epoch = float(matches[-1]["epoch"])
    if not math.isfinite(epoch) or epoch <= 0:
        raise ValueError("Epoch cua checkpoint khong hop le.")
    return int(math.ceil(epoch - 1e-9))


weighted_focal_loss = make_weighted_focal_loss(class_weights, FOCAL_GAMMA)


In [ ]:
training_args = TrainingArguments(
    output_dir=str(RUN_DIR / "checkpoints"),
    run_name=RUN_NAME,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    max_grad_norm=1.0,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    label_names=["labels"],
    remove_unused_columns=True,
    eval_accumulation_steps=16,
    dataloader_num_workers=0,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    push_to_hub=False,
    torch_compile=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_loss_func=weighted_focal_loss,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)
# DistilBERT forward khong nhan num_items_in_batch; loss callback xu ly tham so nay.
trainer.model_accepts_loss_kwargs = False
print("Trainer ready. Cell tiep theo bat dau huan luyen tren Colab.")


## 8. Huấn luyện và chọn checkpoint

Wolf train tối đa 4 epoch; DistilBERT tối đa 8 epoch. Early stopping theo raw Macro F1,
sau đó lấy checkpoint có Macro F1 cao nhất. Training loss giảm không đảm bảo điểm test tăng.
Hết VRAM: giảm batch 8 → 4 và tăng accumulation 4 → 8, giữ effective batch 32.
Lưu OUTPUT_ROOT trên Drive để giữ checkpoint khi Colab ngắt phiên.


In [ ]:
train_result = trainer.train()
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation Macro F1:", trainer.state.best_metric)

# Luu selection model (Wolf) hoac adapter + heads (DistilBERT) truoc refit.
trainer.model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.model.config.to_json_file(str(ARTIFACT_DIR / "base_config.json"))
print("Selection weights saved:", ADAPTER_DIR)


## 9. Đánh giá model tốt nhất

Báo cáo Macro F1, F1 từng lớp, theo ngôn ngữ và confusion matrix đều dùng validation chưa tham gia cập nhật trọng số. Kết quả này là holdout validation; chưa phải điểm Kaggle.


In [ ]:
"""Notebook cell: held-out predictions and diagnostics; no test labels are read."""


def build_prediction_audit(frame, true_ids, scores, raw_lengths, content_budget,
                           label_names, stage="raw", scores_are_probabilities=False):
    """Keep scores tied to row IDs; normalized scores are NOT calibrated confidence.

    Stage-prefixed columns remain interpretable if a downstream report replaces
    ``predicted_label`` with a postprocessed prediction. All rows are returned;
    callers select errors only when writing an error-only artifact.
    """
    required = {"id", "language", "label", "text"}
    if not required.issubset(frame.columns) or not frame["id"].is_unique:
        raise ValueError("Validation audit can cot id/language/label/text va ID duy nhat.")
    if not stage.isidentifier():
        raise ValueError("Audit stage phai la ten hop le, vi du raw hoac final.")
    values = np.asarray(scores, dtype=np.float64)
    truth = np.asarray(true_ids)
    lengths = np.asarray(raw_lengths)
    class_count = len(label_names)
    if len(frame) == 0 or class_count < 2 or len(set(label_names)) != class_count:
        raise ValueError("Validation phai co mau va it nhat hai nhan rieng biet.")
    if values.shape != (len(frame), class_count) or not np.isfinite(values).all():
        raise ValueError("Validation scores sai shape hoac co NaN/Inf.")
    if truth.shape != (len(frame),) or not np.isin(truth, np.arange(class_count)).all():
        raise ValueError("Validation true_ids sai shape hoac ngoai mien nhan.")
    truth = truth.astype(np.int64)
    expected_labels = np.asarray(label_names, dtype=object)[truth]
    if not np.array_equal(expected_labels, frame["label"].to_numpy()):
        raise ValueError("Validation nhan/ID order khong khop true_ids.")
    if (lengths.shape != (len(frame),) or not np.isfinite(lengths).all()
            or (lengths < 0).any() or not np.equal(lengths, np.floor(lengths)).all()):
        raise ValueError("raw_lengths phai la so token nguyen khong am, dung thu tu ID.")
    if not isinstance(content_budget, (int, np.integer)) or content_budget < 1:
        raise ValueError("Content token budget phai la so nguyen duong.")
    if scores_are_probabilities:
        if (values < 0).any() or (values > 1).any():
            raise ValueError("Normalized scores phai nam trong [0, 1].")
        if not np.allclose(values.sum(axis=1), 1.0, atol=1e-6, rtol=0):
            raise ValueError("Moi hang normalized scores phai co tong bang 1.")
        normalized = values
    else:
        shifted = values - values.max(axis=1, keepdims=True)
        normalized = np.exp(shifted)
        normalized /= normalized.sum(axis=1, keepdims=True)
    predicted = values.argmax(axis=1)
    ordered_scores = np.sort(normalized, axis=1)
    rows = frame[["id", "language", "label", "text"]].reset_index(drop=True).copy()
    rows["predicted_label"] = np.asarray(label_names, dtype=object)[predicted]
    rows[f"{stage}_predicted_label"] = rows["predicted_label"]
    rows["raw_tokens"] = lengths.astype(np.int64)
    rows["truncated"] = rows["raw_tokens"] > content_budget
    rows["length_bucket"] = pd.cut(
        rows["raw_tokens"], bins=[-1, 128, 256, 512, 1024, np.inf],
        labels=["0-128", "129-256", "257-512", "513-1024", "1025+"],
    )
    rows[f"{stage}_is_error"] = predicted != truth
    rows[f"{stage}_top_score"] = normalized.max(axis=1)
    rows[f"{stage}_true_class_score"] = normalized[np.arange(len(frame)), truth]
    rows[f"{stage}_score_margin"] = ordered_scores[:, -1] - ordered_scores[:, -2]
    return rows


def summarize_prediction_audit(rows, stage="raw"):
    """Count errors with support, avoiding conclusions from F1 on tiny groups."""
    def aggregate(keys):
        summary = rows.groupby(keys, observed=True, dropna=False).agg(
            support=("id", "size"), errors=(f"{stage}_is_error", "sum"),
        ).reset_index()
        summary["support"] = summary["support"].astype(int)
        summary["errors"] = summary["errors"].astype(int)
        summary["correct"] = summary["support"] - summary["errors"]
        summary["error_rate"] = summary["errors"] / summary["support"]
        return summary

    return (aggregate(["language", "label", "truncated"]),
            aggregate(["length_bucket"]))


valid_result = trainer.predict(valid_dataset, metric_key_prefix="validation")
valid_logits = valid_result.predictions
if isinstance(valid_logits, tuple):
    valid_logits = valid_logits[0]
valid_logits = np.asarray(valid_logits, dtype=np.float32)
valid_true = valid_df["label_id"].to_numpy()
raw_errors = build_prediction_audit(
    valid_df, valid_true, valid_logits, valid_dataset.raw_lengths, budget, LABELS,
)
valid_pred = np.argmax(valid_logits, axis=-1)
report = classification_report(
    valid_true, valid_pred, labels=list(range(len(LABELS))),
    target_names=LABELS, output_dict=True, zero_division=0,
)
report_df = pd.DataFrame(report).T
display(report_df)
report_df.to_csv(ARTIFACT_DIR / "validation_class_report.csv")

language_rows = []
for language in LANGUAGES:
    mask = valid_df["language"].eq(language).to_numpy()
    if not mask.any():
        continue
    f1_values = f1_score(
        valid_true[mask], valid_pred[mask], labels=list(range(len(LABELS))),
        average=None, zero_division=0,
    )
    row = {"language": language, "rows": int(mask.sum()), "macro_f1": float(f1_values.mean())}
    row.update({f"f1_{label}": float(score) for label, score in zip(LABELS, f1_values)})
    row.update({f"support_{label}": int((valid_true[mask] == index).sum())
                for index, label in enumerate(LABELS)})
    language_rows.append(row)
language_report = pd.DataFrame(language_rows)
display(language_report)
language_report.to_csv(ARTIFACT_DIR / "validation_by_language.csv", index=False)

validation_predictions = raw_errors.drop(columns=["text"]).copy()
validation_predictions.to_csv(ARTIFACT_DIR / "validation_predictions.csv", index=False)
raw_errors.to_csv(ARTIFACT_DIR / "validation_prediction_audit_raw.csv", index=False)
raw_errors.loc[raw_errors["raw_is_error"]].to_csv(
    ARTIFACT_DIR / "validation_errors_raw.csv", index=False,
)
validation_group_errors, validation_length_errors = summarize_prediction_audit(raw_errors)
validation_group_errors.to_csv(ARTIFACT_DIR / "validation_errors_by_group_raw.csv", index=False)
validation_length_errors.to_csv(ARTIFACT_DIR / "validation_errors_by_length_raw.csv", index=False)
print("Selection-only errors:", int(raw_errors["raw_is_error"].sum()), "/", len(raw_errors))
print("Softmax scores va top-1/top-2 margin KHONG phai do tin cay da calibration.")
display(validation_group_errors.sort_values(["errors", "support"], ascending=False).head(12))
display(validation_length_errors)
error_preview = raw_errors.loc[raw_errors["raw_is_error"]].sort_values(
    "raw_score_margin", ascending=False,
).head(20).copy()
error_preview["text"] = error_preview["text"].str.slice(0, 180)
display(error_preview[["id", "language", "label", "predicted_label", "raw_tokens",
                       "truncated", "raw_score_margin", "text"]])

matrix = confusion_matrix(valid_true, valid_pred, labels=list(range(len(LABELS))))
pd.DataFrame(matrix, index=LABELS, columns=LABELS).to_csv(
    ARTIFACT_DIR / "validation_confusion_matrix.csv",
)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(matrix, display_labels=LABELS).plot(
    ax=ax, values_format="d", cmap="Blues", xticks_rotation=30, colorbar=False,
)
ax.set_title("Selection validation confusion matrix")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "validation_confusion_matrix.png", dpi=150)
plt.show()
plt.close(fig)
np.save(ARTIFACT_DIR / "validation_logits.npy", valid_logits)
valid_df[["id", "language", "label"]].to_csv(ARTIFACT_DIR / "validation_ids.csv", index=False)


## 10. Dự đoán test và xuất submission

Dự đoán giữ liên kết với test ID, sau đó sắp theo **sample_submission.csv**. File nộp chỉ có hai cột **id,label** và đúng 6.000 dòng. Nhãn mẫu không ảnh hưởng dự đoán.


In [ ]:
test_result = trainer.predict(test_dataset, metric_key_prefix="test")
test_logits = test_result.predictions
if isinstance(test_logits, tuple):
    test_logits = test_logits[0]
if test_logits.shape != (len(test_df), len(LABELS)) or not np.isfinite(test_logits).all():
    raise ValueError("Test logits sai shape hoac co NaN/Inf.")
np.save(ARTIFACT_DIR / "test_logits.npy", test_logits.astype(np.float32))
test_df[["id", "language"]].to_csv(ARTIFACT_DIR / "test_ids.csv", index=False)
print("Da luu logits va ID order. Cell sau chon ensemble/bias tu validation.")


### Hậu xử lý global và thử hiệu chỉnh Portuguese

Giữ quy trình global của V3: blend/bias chỉ bật nếu cross-fit tăng ít nhất 0.002.
Sau đó thử bias nhỏ cho Portuguese ở những dự đoán Benign/Jailbreak sát nhau.
Các fold ngoài đánh giá trên những dòng không tham gia chọn global policy hoặc
bias của chính fold đó. Chỉ dùng hiệu chỉnh mới nếu gain overall ≥ 0.002, gain
Portuguese dương, ít nhất 2/3 fold cải thiện và kiểm tra cuối không làm giảm điểm.
Nếu không đạt, giữ nguyên kết quả global. Bước PT không đổi dự đoán PI/ngôn ngữ khác.

Checkpoint vẫn đã được chọn trên validation này, nên nested cross-fit hậu xử lý
không biến các số trên thành đánh giá model độc lập. Cả điểm raw, global và PT
được lưu riêng; điểm fit trên toàn validation không phải dự báo điểm Kaggle.


In [ ]:
"""Conservative post-processing cell for the RMIT C1 notebook.

Paste this file into one notebook code cell after ``valid_logits`` and
``test_logits`` exist. If the lexical SVM is enabled, its decision-function
outputs must be named ``valid_lexical_scores`` and ``test_lexical_scores`` and
must already use the same class order as ``LABELS``.

This cell deliberately performs no artifact I/O. The notebook's artifact cell
should save the returned configuration, report, predictions, and source logits.
"""

from __future__ import annotations

from itertools import product
from typing import Any, Iterable, Mapping, Optional, Sequence

import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold


EXPECTED_POSTPROCESS_LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]


def _as_score_matrix(name: str, values: Any, rows: Optional[int] = None) -> np.ndarray:
    """Return a finite float64 (rows, 3) score matrix."""
    matrix = np.asarray(values, dtype=np.float64)
    if matrix.ndim != 2 or matrix.shape[1] != len(EXPECTED_POSTPROCESS_LABELS):
        raise ValueError(
            f"{name} must have shape (n, {len(EXPECTED_POSTPROCESS_LABELS)}); "
            f"received {matrix.shape}."
        )
    if rows is not None and matrix.shape[0] != rows:
        raise ValueError(f"{name} must have {rows} rows; received {matrix.shape[0]}.")
    if not np.isfinite(matrix).all():
        raise ValueError(f"{name} contains NaN or Inf.")
    return matrix


def log_softmax_np(scores: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    """Numerically stable log-softmax for a small dense score matrix."""
    temperature = float(temperature)
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("temperature must be finite and positive.")
    scaled = np.asarray(scores, dtype=np.float64) / temperature
    shifted = scaled - scaled.max(axis=1, keepdims=True)
    return shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))


def softmax_from_log_scores(log_scores: np.ndarray) -> np.ndarray:
    """Normalize arbitrary log-scores into probabilities."""
    values = np.asarray(log_scores, dtype=np.float64)
    shifted = values - values.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    return probabilities


def macro_f1(y_true: np.ndarray, y_pred: np.ndarray, class_count: int = 3) -> float:
    """Exact unweighted macro F1, including classes with zero support."""
    truth = np.asarray(y_true, dtype=np.int64)
    predicted = np.asarray(y_pred, dtype=np.int64)
    if truth.shape != predicted.shape or truth.ndim != 1:
        raise ValueError("Expected aligned one-dimensional labels.")
    if not np.isin(truth, np.arange(class_count)).all() or not np.isin(predicted, np.arange(class_count)).all():
        raise ValueError("Class index outside allowed range.")
    matrix = np.bincount(class_count * truth + predicted, minlength=class_count ** 2).reshape(class_count, class_count)
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    values = np.divide(2.0 * matrix.diagonal(), denominator,
                       out=np.zeros(class_count, dtype=np.float64), where=denominator > 0)
    return float(values.mean())


def postprocess_log_scores(
    neural_logits: np.ndarray,
    lexical_scores: Optional[np.ndarray],
    config: Mapping[str, Any],
) -> np.ndarray:
    """Apply a fitted temperature/blend/bias configuration."""
    neural_log_proba = log_softmax_np(
        neural_logits, float(config["neural_temperature"])
    )
    lexical_weight = float(config.get("lexical_weight", 0.0))
    if lexical_weight > 0:
        if lexical_scores is None:
            raise ValueError("The fitted configuration requires lexical scores.")
        lexical_log_proba = log_softmax_np(
            lexical_scores, float(config["lexical_temperature"])
        )
        combined = (
            (1.0 - lexical_weight) * neural_log_proba
            + lexical_weight * lexical_log_proba
        )
    else:
        combined = neural_log_proba

    bias = np.asarray(config.get("class_bias", [0.0, 0.0, 0.0]), dtype=np.float64)
    if bias.shape != (neural_log_proba.shape[1],) or not np.isfinite(bias).all():
        raise ValueError("class_bias has an invalid shape or value.")
    return combined + bias[None, :]


def _config_simplicity_key(config: Mapping[str, Any]) -> tuple[Any, ...]:
    """Prefer no bias, then neural-only, then values closest to identity."""
    bias = np.asarray(config["class_bias"], dtype=np.float64)
    bias_norm = float(np.dot(bias, bias))
    lexical_weight = float(config["lexical_weight"])
    lexical_temperature = config.get("lexical_temperature")
    return (
        int(bias_norm > 1e-15),
        int(lexical_weight > 1e-15),
        bias_norm,
        lexical_weight,
        abs(float(np.log(float(config["neural_temperature"])))),
        0.0
        if lexical_temperature is None
        else abs(float(np.log(float(lexical_temperature)))),
    )


def fit_postprocessor(
    neural_logits: np.ndarray,
    y_true: np.ndarray,
    lexical_scores: Optional[np.ndarray] = None,
    *,
    neural_temperatures: Sequence[float] = (1.0, 2.0),
    lexical_temperatures: Sequence[float] = (0.5, 1.0),
    lexical_weights: Sequence[float] = (0.0, 0.25, 0.50, 0.75, 0.90),
    bias_values: Sequence[float] = (-0.30, -0.15, 0.0, 0.15, 0.30),
    jailbreak_bias_values: Optional[Sequence[float]] = None,
    injection_bias_values: Optional[Sequence[float]] = None,
    bias_l2: float = 0.002,
    tie_tolerance: float = 1e-12,
) -> dict[str, Any]:
    """Fit a small global post-processing grid on the supplied rows.

    BENIGN bias is fixed to zero because adding the same constant to all class
    scores cannot change a prediction. Only two bias coordinates are searched.
    Candidate matrices are evaluated one at a time to keep memory bounded.
    """
    neural_logits = _as_score_matrix("neural_logits", neural_logits)
    y_true = np.asarray(y_true, dtype=np.int64)
    if y_true.shape != (neural_logits.shape[0],):
        raise ValueError("y_true length does not match neural_logits.")
    if not np.isin(y_true, np.arange(neural_logits.shape[1])).all():
        raise ValueError("y_true contains a class outside [0, 2].")
    if lexical_scores is not None:
        lexical_scores = _as_score_matrix(
            "lexical_scores", lexical_scores, neural_logits.shape[0]
        )

    neural_temperatures = tuple(float(value) for value in neural_temperatures)
    lexical_temperatures = tuple(float(value) for value in lexical_temperatures)
    lexical_weights = tuple(float(value) for value in lexical_weights)
    bias_values = tuple(float(value) for value in bias_values)
    jailbreak_grid = tuple(bias_values if jailbreak_bias_values is None else jailbreak_bias_values)
    injection_grid = tuple(bias_values if injection_bias_values is None else injection_bias_values)
    if any(0.0 not in grid or not np.isfinite(grid).all() for grid in (jailbreak_grid, injection_grid)):
        raise ValueError("Moi bias grid can 0 va cac gia tri huu han.")
    if 1.0 not in neural_temperatures or 0.0 not in lexical_weights or 0.0 not in bias_values:
        raise ValueError("The conservative grid must include temperature=1, weight=0, and bias=0.")
    if any(value <= 0 for value in neural_temperatures + lexical_temperatures):
        raise ValueError("All temperatures must be positive.")
    if any(value < 0 or value >= 1 for value in lexical_weights):
        raise ValueError("lexical_weights must be in [0, 1).")
    if bias_l2 < 0:
        raise ValueError("bias_l2 must be nonnegative.")

    neural_cache = {
        temperature: log_softmax_np(neural_logits, temperature)
        for temperature in neural_temperatures
    }
    lexical_cache = (
        {
            temperature: log_softmax_np(lexical_scores, temperature)
            for temperature in lexical_temperatures
        }
        if lexical_scores is not None
        else {}
    )

    best_config: Optional[dict[str, Any]] = None
    best_objective = -np.inf
    best_score = -np.inf
    candidate_count = 0

    for neural_temperature in neural_temperatures:
        neural_log_proba = neural_cache[neural_temperature]
        weights_to_try: Iterable[float] = (
            lexical_weights if lexical_scores is not None else (0.0,)
        )
        for lexical_weight in weights_to_try:
            temperatures_to_try: Iterable[Optional[float]] = (
                (None,) if lexical_weight == 0.0 else lexical_temperatures
            )
            for lexical_temperature in temperatures_to_try:
                if lexical_weight == 0.0:
                    base_scores = neural_log_proba
                else:
                    lexical_log_proba = lexical_cache[float(lexical_temperature)]
                    base_scores = (
                        (1.0 - lexical_weight) * neural_log_proba
                        + lexical_weight * lexical_log_proba
                    )

                for jailbreak_bias, injection_bias in product(jailbreak_grid, injection_grid):
                    bias = np.array([0.0, jailbreak_bias, injection_bias], dtype=np.float64)
                    predictions = np.argmax(base_scores + bias[None, :], axis=1)
                    score = macro_f1(y_true, predictions, base_scores.shape[1])
                    bias_norm = float(np.dot(bias, bias))
                    objective = score - float(bias_l2) * bias_norm
                    config = {
                        "neural_temperature": float(neural_temperature),
                        "lexical_temperature": (
                            None if lexical_temperature is None else float(lexical_temperature)
                        ),
                        "lexical_weight": float(lexical_weight),
                        "class_bias": bias.tolist(),
                    }
                    candidate_count += 1

                    is_better = objective > best_objective + tie_tolerance
                    is_tied = abs(objective - best_objective) <= tie_tolerance
                    if is_better or (
                        is_tied
                        and (
                            best_config is None
                            or _config_simplicity_key(config)
                            < _config_simplicity_key(best_config)
                        )
                    ):
                        best_config = config
                        best_objective = float(objective)
                        best_score = float(score)

    if best_config is None:
        raise RuntimeError("Post-processing grid unexpectedly produced no candidate.")
    best_config.update(
        {
            "fit_macro_f1": best_score,
            "penalized_objective": best_objective,
            "bias_l2": float(bias_l2),
            "candidate_count": int(candidate_count),
        }
    )
    return best_config


def cross_fitted_postprocess(
    neural_logits: np.ndarray,
    y_true: np.ndarray,
    strata: np.ndarray,
    lexical_scores: Optional[np.ndarray] = None,
    *,
    n_splits: int = 3,
    random_state: int = 42,
    fit_kwargs: Optional[Mapping[str, Any]] = None,
) -> tuple[np.ndarray, list[dict[str, Any]]]:
    """Select post-processing on two folds and predict the held-out fold."""
    neural_logits = _as_score_matrix("neural_logits", neural_logits)
    y_true = np.asarray(y_true, dtype=np.int64)
    strata = np.asarray(strata, dtype=str)
    if y_true.shape != (neural_logits.shape[0],) or strata.shape != y_true.shape:
        raise ValueError("y_true/strata length does not match neural_logits.")
    if lexical_scores is not None:
        lexical_scores = _as_score_matrix(
            "lexical_scores", lexical_scores, neural_logits.shape[0]
        )

    unique_strata, stratum_counts = np.unique(strata, return_counts=True)
    if len(unique_strata) < neural_logits.shape[1]:
        raise ValueError("Too few strata for cross-fitted post-processing.")
    if int(stratum_counts.min()) < int(n_splits):
        raise ValueError(
            f"Every language::label stratum needs at least {n_splits} rows; "
            f"smallest has {int(stratum_counts.min())}."
        )

    splitter = StratifiedKFold(
        n_splits=int(n_splits), shuffle=True, random_state=int(random_state)
    )
    oof_predictions = np.full(y_true.shape, -1, dtype=np.int64)
    fold_reports: list[dict[str, Any]] = []
    fit_kwargs = dict(fit_kwargs or {})

    for fold, (fit_index, holdout_index) in enumerate(
        splitter.split(np.zeros(y_true.shape[0]), strata), start=1
    ):
        fit_lexical = None if lexical_scores is None else lexical_scores[fit_index]
        holdout_lexical = None if lexical_scores is None else lexical_scores[holdout_index]
        config = fit_postprocessor(
            neural_logits[fit_index],
            y_true[fit_index],
            fit_lexical,
            **fit_kwargs,
        )
        holdout_log_scores = postprocess_log_scores(
            neural_logits[holdout_index], holdout_lexical, config
        )
        holdout_predictions = np.argmax(holdout_log_scores, axis=1)
        oof_predictions[holdout_index] = holdout_predictions
        raw_predictions = np.argmax(neural_logits[holdout_index], axis=1)
        raw_score = macro_f1(y_true[holdout_index], raw_predictions)
        post_score = macro_f1(y_true[holdout_index], holdout_predictions)
        fold_reports.append(
            {
                "fold": int(fold),
                "fit_rows": int(fit_index.size),
                "holdout_rows": int(holdout_index.size),
                "holdout_raw_neural_macro_f1": raw_score,
                "holdout_postprocess_macro_f1": post_score,
                "holdout_gain": float(post_score - raw_score),
                "selected_config": config,
            }
        )

    if (oof_predictions < 0).any():
        raise RuntimeError("Some validation rows did not receive a cross-fitted prediction.")
    return oof_predictions, fold_reports


def run_postprocessing(
    valid_logits: np.ndarray,
    test_logits: np.ndarray,
    valid_df: Any,
    labels: Sequence[str],
    valid_lexical_scores: Optional[np.ndarray] = None,
    test_lexical_scores: Optional[np.ndarray] = None,
    *,
    split_seed: int = 42,
    min_postprocess_gain: float = 0.002,
    n_splits: int = 3,
    fit_kwargs: Optional[Mapping[str, Any]] = None,
) -> dict[str, Any]:
    """Run cross-fitted selection, its safety gate, and final all-valid fit."""
    if list(labels) != EXPECTED_POSTPROCESS_LABELS:
        raise ValueError(
            f"LABELS must be exactly {EXPECTED_POSTPROCESS_LABELS}; received {list(labels)}."
        )
    valid_logits = _as_score_matrix("valid_logits", valid_logits, len(valid_df))
    test_logits = _as_score_matrix("test_logits", test_logits)
    if min_postprocess_gain < 0:
        raise ValueError("min_postprocess_gain must be nonnegative.")

    lexical_enabled = valid_lexical_scores is not None or test_lexical_scores is not None
    if lexical_enabled and (valid_lexical_scores is None or test_lexical_scores is None):
        raise ValueError(
            "valid_lexical_scores and test_lexical_scores must both be provided or both be None."
        )
    if lexical_enabled:
        valid_lexical_scores = _as_score_matrix(
            "valid_lexical_scores", valid_lexical_scores, valid_logits.shape[0]
        )
        test_lexical_scores = _as_score_matrix(
            "test_lexical_scores", test_lexical_scores, test_logits.shape[0]
        )

    required_columns = {"id", "language", "label_id"}
    missing_columns = required_columns - set(valid_df.columns)
    if missing_columns:
        raise ValueError(f"valid_df is missing columns: {sorted(missing_columns)}.")
    if valid_df["id"].duplicated().any():
        raise ValueError("valid_df IDs must be unique.")
    y_true = valid_df["label_id"].to_numpy(dtype=np.int64)
    if not np.isin(y_true, np.arange(len(labels))).all():
        raise ValueError("valid_df.label_id contains a class outside [0, 2].")
    languages = valid_df["language"].astype(str).to_numpy()
    strata = np.asarray(
        [f"{language}::{label_id}" for language, label_id in zip(languages, y_true)],
        dtype=str,
    )

    raw_valid_pred = np.argmax(valid_logits, axis=1)
    raw_macro = macro_f1(y_true, raw_valid_pred, len(labels))
    oof_predictions, fold_reports = cross_fitted_postprocess(
        valid_logits,
        y_true,
        strata,
        valid_lexical_scores,
        n_splits=n_splits,
        random_state=split_seed,
        fit_kwargs=fit_kwargs,
    )
    oof_macro = macro_f1(y_true, oof_predictions, len(labels))
    oof_gain = float(oof_macro - raw_macro)
    gate_passed = bool(oof_gain >= float(min_postprocess_gain))

    shared_validation_warning = (
        "Cross-fitting protects the post-processing comparison from fitting and scoring each "
        "bias/blend on the same rows. It is still not an independent model estimate because "
        "the checkpoint and other hyperparameters were selected using this validation set."
    )
    all_valid_fit_warning = (
        "The all-validation tuned Macro F1 is an in-sample diagnostic only; use the "
        "cross-fitted score for the post-processing gate."
    )

    if gate_passed:
        fitted_config = fit_postprocessor(
            valid_logits,
            y_true,
            valid_lexical_scores,
            **dict(fit_kwargs or {}),
        )
        final_valid_log_scores = postprocess_log_scores(
            valid_logits, valid_lexical_scores, fitted_config
        )
        final_test_log_scores = postprocess_log_scores(
            test_logits, test_lexical_scores, fitted_config
        )
        final_valid_proba = softmax_from_log_scores(final_valid_log_scores)
        final_test_proba = softmax_from_log_scores(final_test_log_scores)
        final_valid_pred = np.argmax(final_valid_proba, axis=1)
        final_test_pred = np.argmax(final_test_proba, axis=1)
        full_valid_macro = macro_f1(y_true, final_valid_pred, len(labels))
        postprocess_config = {
            "enabled": True,
            "mode": "global_log_probability_blend_and_bias",
            "lexical_enabled": bool(lexical_enabled),
            "minimum_oof_gain": float(min_postprocess_gain),
            "parameters": fitted_config,
        }
    else:
        final_valid_log_scores = log_softmax_np(valid_logits, 1.0)
        final_test_log_scores = log_softmax_np(test_logits, 1.0)
        final_valid_proba = np.exp(final_valid_log_scores)
        final_test_proba = np.exp(final_test_log_scores)
        final_valid_pred = raw_valid_pred.copy()
        final_test_pred = np.argmax(test_logits, axis=1)
        full_valid_macro = raw_macro
        postprocess_config = {
            "enabled": False,
            "mode": "raw_neural",
            "lexical_enabled": bool(lexical_enabled),
            "minimum_oof_gain": float(min_postprocess_gain),
            "parameters": {
                "neural_temperature": 1.0,
                "lexical_temperature": None,
                "lexical_weight": 0.0,
                "class_bias": [0.0, 0.0, 0.0],
            },
        }

    selection_report = {
        "gate_passed": gate_passed,
        "lexical_enabled": bool(lexical_enabled),
        "validation_rows": int(valid_logits.shape[0]),
        "cross_fit_folds": int(n_splits),
        "raw_neural_macro_f1": raw_macro,
        "cross_fitted_postprocess_macro_f1": oof_macro,
        "cross_fitted_gain": oof_gain,
        "minimum_required_gain": float(min_postprocess_gain),
        "all_validation_tuned_macro_f1": full_valid_macro,
        "folds": fold_reports,
        "warnings": [shared_validation_warning, all_valid_fit_warning],
        "language_specific_bias": False,
    }

    return {
        "final_valid_pred": final_valid_pred,
        "final_test_pred": final_test_pred,
        "final_valid_proba": final_valid_proba,
        "final_test_proba": final_test_proba,
        "final_valid_scores": final_valid_log_scores,
        "final_test_scores": final_test_log_scores,
        "postprocess_oof_pred": oof_predictions,
        "POSTPROCESS_CONFIG": postprocess_config,
        "POSTPROCESS_SELECTION_REPORT": selection_report,
    }


# When pasted as a notebook cell, publish the requested variables. Importing
# this file as a module does not execute the notebook-specific branch.
_required_notebook_names = {"valid_logits", "test_logits", "valid_df", "LABELS"}
if _required_notebook_names.issubset(globals()):
    MIN_POSTPROCESS_GAIN = float(globals().get("MIN_POSTPROCESS_GAIN", 0.002))
    POSTPROCESS_SPLIT_SEED = int(
        globals().get("SPLIT_SEED", globals().get("SEED", 42))
    )
    _postprocess_result = run_postprocessing(
        valid_logits=globals()["valid_logits"],
        test_logits=globals()["test_logits"],
        valid_df=globals()["valid_df"],
        labels=globals()["LABELS"],
        valid_lexical_scores=globals().get("valid_lexical_scores"),
        test_lexical_scores=globals().get("test_lexical_scores"),
        split_seed=POSTPROCESS_SPLIT_SEED,
        min_postprocess_gain=MIN_POSTPROCESS_GAIN,
        n_splits=3,
        fit_kwargs=POSTPROCESS_FIT_KWARGS,
    )
    globals().update(_postprocess_result)
    print(
        "Post-processing:",
        "enabled" if POSTPROCESS_CONFIG["enabled"] else "raw neural retained",
        "| raw Macro F1 =",
        round(POSTPROCESS_SELECTION_REPORT["raw_neural_macro_f1"], 6),
        "| cross-fitted Macro F1 =",
        round(POSTPROCESS_SELECTION_REPORT["cross_fitted_postprocess_macro_f1"], 6),
        "| gain =",
        round(POSTPROCESS_SELECTION_REPORT["cross_fitted_gain"], 6),
    )

POSTPROCESS_SELECTION_REPORT["raw_lexical_macro_f1"] = (
    macro_f1(valid_true, valid_lexical_scores.argmax(axis=1))
    if valid_lexical_scores is not None else None
)
POSTPROCESS_CONFIG["labels"] = list(LABELS)
POSTPROCESS_CONFIG["score_interpretation"] = (
    "Normalized tempered log-score blend, tuned for Macro F1; not calibrated confidence."
)
POSTPROCESS_CONFIG["search_grid"] = {
    "neural_temperatures": [1.0, 2.0], "lexical_temperatures": [0.5, 1.0],
    "lexical_weights": [0.0, 0.25, 0.50, 0.75, 0.90],
    **POSTPROCESS_FIT_KWARGS,
}
print("Final parameters:", POSTPROCESS_CONFIG["parameters"])

"""Optional Portuguese BENIGN/JAILBREAK correction after the global policy.

This cell relies on the global post-processing helpers defined in the preceding
cell. It does not run training, read labels from test, or execute on import.
"""

from copy import deepcopy
from typing import Any, Mapping, Optional, Sequence

import numpy as np
from sklearn.model_selection import StratifiedKFold


def _identity_postprocess_parameters() -> dict[str, Any]:
    return {
        "neural_temperature": 1.0,
        "lexical_temperature": None,
        "lexical_weight": 0.0,
        "class_bias": [0.0, 0.0, 0.0],
    }


def apply_portuguese_pair_bias(
    log_scores: np.ndarray,
    languages: Sequence[str],
    bias: float = 0.0,
    max_margin: float = 0.75,
) -> np.ndarray:
    """Change only low-margin Portuguese BENIGN/JAILBREAK decisions.

    Positive bias favors JAILBREAK; negative bias favors BENIGN. Skip a proposed
    change if it would make PROMPT_INJECTION win. An existing PROMPT_INJECTION
    winner and every non-Portuguese row remain unchanged in score space.
    """
    scores = _as_score_matrix("pair_log_scores", log_scores)
    languages = np.asarray(languages, dtype=str)
    if languages.shape != (len(scores),):
        raise ValueError("languages must contain exactly one value per score row.")
    if not np.isfinite(bias) or not np.isfinite(max_margin) or max_margin <= 0:
        raise ValueError("Pair bias must be finite and max_margin must be positive.")
    adjusted = scores.copy()
    if float(bias) == 0.0:
        return adjusted
    original_pred = scores.argmax(axis=1)
    eligible = (
        (languages == "pt")
        & (original_pred != 2)
        & (np.abs(scores[:, 1] - scores[:, 0]) <= float(max_margin))
    )
    adjusted[eligible, 1] += float(bias)
    new_pi = eligible & (adjusted.argmax(axis=1) == 2)
    adjusted[new_pi] = scores[new_pi]
    return adjusted


def apply_selected_postprocess(
    neural_logits: np.ndarray,
    lexical_scores: Optional[np.ndarray],
    policy: Mapping[str, Any],
    languages: Optional[Sequence[str]] = None,
) -> np.ndarray:
    """Apply frozen global and optional pair policies, including after refit."""
    scores = postprocess_log_scores(neural_logits, lexical_scores, policy["parameters"])
    pair = policy.get("portuguese_pair", {})
    if pair.get("enabled", False):
        if languages is None:
            raise ValueError("Frozen Portuguese pair policy requires row languages.")
        scores = apply_portuguese_pair_bias(
            scores, languages, float(pair["bias"]), float(pair["max_margin"])
        )
    return scores


def _fit_portuguese_pair_bias(
    scores: np.ndarray,
    y_true: np.ndarray,
    languages: np.ndarray,
    *,
    bias_values: Sequence[float],
    max_margin: float,
    bias_l2: float,
    min_pt_class_support: int,
) -> dict[str, Any]:
    """Fit one shrunk scalar on supplied development rows only."""
    baseline_score = macro_f1(y_true, scores.argmax(axis=1))
    support = [int(((languages == "pt") & (y_true == label)).sum()) for label in (0, 1)]
    best = {
        "bias": 0.0,
        "fit_macro_f1": baseline_score,
        "penalized_objective": baseline_score,
        "pt_benign_support": support[0],
        "pt_jailbreak_support": support[1],
    }
    if min(support) < min_pt_class_support:
        best["reason"] = "insufficient_portuguese_pair_support"
        return best
    for bias in bias_values:
        predicted = apply_portuguese_pair_bias(scores, languages, bias, max_margin).argmax(axis=1)
        score = macro_f1(y_true, predicted)
        objective = score - bias_l2 * float(bias) ** 2
        better = objective > best["penalized_objective"] + 1e-12
        tied = abs(objective - best["penalized_objective"]) <= 1e-12
        if better or (tied and abs(bias) < abs(best["bias"])):
            best.update(bias=float(bias), fit_macro_f1=float(score),
                        penalized_objective=float(objective))
    return best


def _crossfit_global_log_scores(
    logits: np.ndarray,
    y_true: np.ndarray,
    strata: np.ndarray,
    lexical: Optional[np.ndarray],
    *,
    n_splits: int,
    seed: int,
    fit_kwargs: Mapping[str, Any],
) -> tuple[np.ndarray, list[dict[str, Any]]]:
    """Return scores for rows excluded from their global-policy fitting fold."""
    scores = np.full(logits.shape, np.nan, dtype=np.float64)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for fit_index, held_index in splitter.split(np.zeros(len(y_true)), strata):
        params = fit_postprocessor(
            logits[fit_index], y_true[fit_index],
            None if lexical is None else lexical[fit_index], **dict(fit_kwargs),
        )
        scores[held_index] = postprocess_log_scores(
            logits[held_index], None if lexical is None else lexical[held_index], params
        )
        splits.append({"fit_indices": fit_index.tolist(), "holdout_indices": held_index.tolist()})
    if not np.isfinite(scores).all():
        raise RuntimeError("Global cross-fitting left a validation row unpredicted.")
    return scores, splits


def run_portuguese_pair_tuning(
    valid_logits: np.ndarray,
    test_logits: np.ndarray,
    valid_df: Any,
    test_languages: Sequence[str],
    base_result: Mapping[str, Any],
    valid_lexical_scores: Optional[np.ndarray] = None,
    test_lexical_scores: Optional[np.ndarray] = None,
    *,
    split_seed: int = 42,
    min_postprocess_gain: float = 0.002,
    min_pair_gain: float = 0.002,
    n_splits: int = 3,
    global_fit_kwargs: Optional[Mapping[str, Any]] = None,
    bias_values: Sequence[float] = (-0.30, -0.15, 0.0, 0.15, 0.30),
    max_margin: float = 0.75,
    bias_l2: float = 0.002,
    min_pt_class_support: int = 20,
    verbose: bool = True,
) -> dict[str, Any]:
    """Evaluate a pair correction against an independently fitted fold baseline.

    Each outer training portion independently chooses its global blend/bias via
    inner cross-fitting. The pair bias is fit using inner out-of-fold scores and
    is scored on the untouched outer holdout. Both compared predictions come
    from the SAME fold-specific global pipeline. The existing deployed global
    gate is retained. Cross-fitting does not undo earlier checkpoint selection
    on this validation set, so these scores are not independent model estimates.
    """
    logits = _as_score_matrix("valid_logits", valid_logits, len(valid_df))
    test = _as_score_matrix("test_logits", test_logits)
    required = {"id", "language", "label_id"}
    if not required.issubset(valid_df.columns) or valid_df["id"].duplicated().any():
        raise ValueError("valid_df needs unique id plus language and label_id columns.")
    y_true = valid_df["label_id"].to_numpy(dtype=np.int64)
    languages = valid_df["language"].astype(str).to_numpy()
    test_languages = np.asarray(test_languages, dtype=str)
    if test_languages.shape != (len(test),):
        raise ValueError("test_languages must align with test_logits.")
    if not np.isin(y_true, [0, 1, 2]).all():
        raise ValueError("Validation labels must be class indices 0, 1, 2.")
    if (valid_lexical_scores is None) != (test_lexical_scores is None):
        raise ValueError("Provide both validation and test lexical scores, or neither.")
    lexical = (None if valid_lexical_scores is None else
               _as_score_matrix("valid_lexical_scores", valid_lexical_scores, len(logits)))
    test_lexical = (None if test_lexical_scores is None else
                    _as_score_matrix("test_lexical_scores", test_lexical_scores, len(test)))
    bias_values = tuple(float(value) for value in bias_values)
    if not bias_values or 0.0 not in bias_values or not np.isfinite(bias_values).all():
        raise ValueError("Pair bias grid must be finite and contain zero.")
    if any(abs(value) > 0.30 for value in bias_values):
        raise ValueError("Pair correction is limited to biases between -0.30 and +0.30.")
    if n_splits < 3 or min_pair_gain <= 0 or min_postprocess_gain < 0 or bias_l2 < 0:
        raise ValueError("Need >=3 folds, positive pair gain, nonnegative global gain and penalty.")
    if not np.isfinite(max_margin) or max_margin <= 0 or min_pt_class_support < 1:
        raise ValueError("Need positive max_margin and class-support threshold.")
    fit_kwargs = dict(global_fit_kwargs or {})
    pair_fit_kwargs = dict(bias_values=bias_values, max_margin=max_margin,
                           bias_l2=bias_l2, min_pt_class_support=min_pt_class_support)
    result = dict(base_result)
    policy = deepcopy(base_result["POSTPROCESS_CONFIG"])
    selection_report = deepcopy(base_result["POSTPROCESS_SELECTION_REPORT"])
    baseline_valid_scores = apply_selected_postprocess(logits, lexical, policy, languages)
    baseline_test_scores = apply_selected_postprocess(test, test_lexical, policy, test_languages)
    if policy.get("portuguese_pair", {}).get("enabled", False):
        raise ValueError("Pair fitting must start from the original global-only policy.")
    np.testing.assert_array_equal(baseline_valid_scores.argmax(axis=1), base_result["final_valid_pred"])
    pair_policy = {
        "enabled": False, "bias": 0.0, "max_margin": float(max_margin),
        "minimum_outer_gain": float(min_pair_gain), "bias_grid": list(bias_values),
        "bias_l2": float(bias_l2), "minimum_pt_class_support": int(min_pt_class_support),
    }
    policy["portuguese_pair"] = pair_policy
    report = {
        "enabled": False,
        "global_policy_retained": True,
        "comparison": "pair correction versus the same independently selected global pipeline",
        "outer_folds": int(n_splits),
        "minimum_required_gain": float(min_pair_gain),
        "warnings": [
            "Outer held-out labels never fit that fold's global gate or pair bias.",
            "The checkpoint was selected on this validation set; this is not an independent model estimate.",
            "All-validation scores are fitting diagnostics, not evidence of expected Kaggle performance.",
        ],
    }
    strata = np.asarray([f"{lang}::{label}" for lang, label in zip(languages, y_true)])
    _, counts = np.unique(strata, return_counts=True)
    # Every inner fold must still be stratifiable after removing one outer fold.
    enough_strata = len(counts) >= 3 and counts.min() - np.ceil(counts.min() / n_splits) >= n_splits
    enough_pt = all(int(((languages == "pt") & (y_true == label)).sum()) >= min_pt_class_support
                    for label in (0, 1))
    if not enough_strata or not enough_pt:
        report["reason"] = "insufficient_rows_for_nested_crossfit_or_portuguese_support"
        selection_report["portuguese_pair"] = report
        result.update(POSTPROCESS_CONFIG=policy, POSTPROCESS_SELECTION_REPORT=selection_report)
        if verbose:
            print("Portuguese pair adjustment: retained global baseline; insufficient validation support.")
        return result

    baseline_oof = np.full(len(logits), -1, dtype=np.int64)
    candidate_oof = np.full(len(logits), -1, dtype=np.int64)
    outer_reports = []
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=split_seed)
    for fold, (fit_index, held_index) in enumerate(splitter.split(np.zeros(len(logits)), strata), 1):
        fit_lexical = None if lexical is None else lexical[fit_index]
        inner_scores, inner_splits = _crossfit_global_log_scores(
            logits[fit_index], y_true[fit_index], strata[fit_index], fit_lexical,
            n_splits=n_splits, seed=split_seed + fold, fit_kwargs=fit_kwargs,
        )
        raw_fit_score = macro_f1(y_true[fit_index], logits[fit_index].argmax(axis=1))
        inner_gain = macro_f1(y_true[fit_index], inner_scores.argmax(axis=1)) - raw_fit_score
        global_enabled = bool(inner_gain >= min_postprocess_gain)
        if global_enabled:
            fold_params = fit_postprocessor(logits[fit_index], y_true[fit_index], fit_lexical,
                                            **fit_kwargs)
            pair_fit_scores = inner_scores
        else:
            fold_params = _identity_postprocess_parameters()
            pair_fit_scores = log_softmax_np(logits[fit_index])
        pair_fit = _fit_portuguese_pair_bias(
            pair_fit_scores, y_true[fit_index], languages[fit_index], **pair_fit_kwargs
        )
        held_scores = postprocess_log_scores(
            logits[held_index], None if lexical is None else lexical[held_index], fold_params
        )
        base_pred = held_scores.argmax(axis=1)
        candidate_pred = apply_portuguese_pair_bias(
            held_scores, languages[held_index], pair_fit["bias"], max_margin
        ).argmax(axis=1)
        baseline_oof[held_index] = base_pred
        candidate_oof[held_index] = candidate_pred
        base_score = macro_f1(y_true[held_index], base_pred)
        candidate_score = macro_f1(y_true[held_index], candidate_pred)
        outer_reports.append({
            "fold": fold, "fit_indices": fit_index.tolist(), "holdout_indices": held_index.tolist(),
            "inner_splits_local_to_fit_indices": inner_splits,
            "global_gate_passed": global_enabled, "inner_global_gain": float(inner_gain),
            "pair_fit": pair_fit, "baseline_macro_f1": base_score,
            "candidate_macro_f1": candidate_score, "gain": float(candidate_score - base_score),
        })
        if verbose:
            print(f"Portuguese pair fold {fold}/{n_splits}: bias={pair_fit['bias']:+.2f}, "
                  f"held-out gain={candidate_score - base_score:+.6f}")
    if (baseline_oof < 0).any() or (candidate_oof < 0).any():
        raise RuntimeError("Nested cross-fitting did not predict every validation row.")
    preserved = (languages != "pt") | (baseline_oof == 2)
    if not np.array_equal(candidate_oof[preserved], baseline_oof[preserved]):
        raise AssertionError("Pair adjustment changed a protected language or PI winner.")
    if not np.array_equal(candidate_oof == 2, baseline_oof == 2):
        raise AssertionError("Pair adjustment changed PROMPT_INJECTION membership.")
    baseline_macro = macro_f1(y_true, baseline_oof)
    candidate_macro = macro_f1(y_true, candidate_oof)
    outer_gain = candidate_macro - baseline_macro
    pt_mask = languages == "pt"
    pt_gain = macro_f1(y_true[pt_mask], candidate_oof[pt_mask]) - macro_f1(y_true[pt_mask], baseline_oof[pt_mask])
    positive_folds = sum(fold["gain"] > 1e-12 for fold in outer_reports)
    required_positive = int(np.ceil(2 * n_splits / 3))
    outer_gate = bool(outer_gain >= min_pair_gain and pt_gain > 0 and positive_folds >= required_positive)

    # Fit the deployed scalar using OOF global scores. If the original global
    # gate failed, fit to raw scores so a rejected blend cannot be reintroduced.
    if base_result["POSTPROCESS_CONFIG"]["enabled"]:
        final_fit_scores, _ = _crossfit_global_log_scores(
            logits, y_true, strata, lexical, n_splits=n_splits,
            seed=split_seed, fit_kwargs=fit_kwargs,
        )
    else:
        final_fit_scores = log_softmax_np(logits)
    final_pair = _fit_portuguese_pair_bias(final_fit_scores, y_true, languages, **pair_fit_kwargs)
    candidate_valid_scores = apply_portuguese_pair_bias(
        baseline_valid_scores, languages, final_pair["bias"], max_margin
    )
    diagnostic_base = macro_f1(y_true, baseline_valid_scores.argmax(axis=1))
    diagnostic_candidate = macro_f1(y_true, candidate_valid_scores.argmax(axis=1))
    diagnostic_gain = diagnostic_candidate - diagnostic_base
    enabled = bool(outer_gate and final_pair["bias"] != 0 and diagnostic_gain > 0)
    pair_policy.update(enabled=enabled, bias=float(final_pair["bias"]) if enabled else 0.0)
    report.update({
        "enabled": enabled, "outer_gate_passed": outer_gate,
        "nested_baseline_macro_f1": baseline_macro, "nested_candidate_macro_f1": candidate_macro,
        "nested_gain": float(outer_gain), "portuguese_nested_gain": float(pt_gain),
        "positive_folds": int(positive_folds), "required_positive_folds": required_positive,
        "folds": outer_reports, "final_fit": final_pair,
        "all_validation_baseline_macro_f1": diagnostic_base,
        "all_validation_candidate_macro_f1": diagnostic_candidate,
        "all_validation_diagnostic_gain": float(diagnostic_gain),
        "reason": "passed_nested_and_sanity_gates" if enabled else "global_baseline_retained",
    })
    selection_report["portuguese_pair"] = report
    selection_report["language_specific_bias"] = enabled
    # Preserve enabled/mode: these describe the ORIGINAL GLOBAL gate. Consumers
    # must call apply_selected_postprocess even when that gate is False.
    final_valid_scores = apply_selected_postprocess(logits, lexical, policy, languages)
    final_test_scores = apply_selected_postprocess(test, test_lexical, policy, test_languages)
    result.update(
        final_valid_scores=final_valid_scores, final_test_scores=final_test_scores,
        final_valid_proba=softmax_from_log_scores(final_valid_scores),
        final_test_proba=softmax_from_log_scores(final_test_scores),
        final_valid_pred=final_valid_scores.argmax(axis=1),
        final_test_pred=final_test_scores.argmax(axis=1),
        portuguese_pair_baseline_oof_pred=baseline_oof,
        portuguese_pair_candidate_oof_pred=candidate_oof,
        POSTPROCESS_CONFIG=policy, POSTPROCESS_SELECTION_REPORT=selection_report,
    )
    if not enabled:
        # Keep original arrays exactly, including their probability rounding.
        for name in ("final_valid_pred", "final_test_pred", "final_valid_scores",
                     "final_test_scores", "final_valid_proba", "final_test_proba"):
            result[name] = base_result[name]
    if verbose:
        print("Portuguese pair adjustment:", "enabled" if enabled else "global baseline retained",
              f"| nested gain={outer_gain:+.6f} | positive folds={positive_folds}/{n_splits}")
    return result


# Append after the original global block. Preserve both prediction sets so the
# exported pair candidate can be compared with the original selected policy.
if {"_postprocess_result", "valid_logits", "test_logits", "test_df"}.issubset(globals()):
    GLOBAL_POSTPROCESS_CONFIG = deepcopy(POSTPROCESS_CONFIG)
    GLOBAL_POSTPROCESS_SELECTION_REPORT = deepcopy(POSTPROCESS_SELECTION_REPORT)
    global_valid_pred = final_valid_pred.copy()
    global_test_pred = final_test_pred.copy()
    global_test_proba = final_test_proba.copy()
    if bool(globals().get("ENABLE_PT_PAIR_TUNING", False)):
        _postprocess_result = run_portuguese_pair_tuning(
            valid_logits=valid_logits,
            test_logits=test_logits,
            valid_df=valid_df,
            test_languages=test_df["language"].astype(str).to_numpy(),
            base_result=_postprocess_result,
            valid_lexical_scores=globals().get("valid_lexical_scores"),
            test_lexical_scores=globals().get("test_lexical_scores"),
            split_seed=int(SPLIT_SEED),
            min_postprocess_gain=float(MIN_POSTPROCESS_GAIN),
            min_pair_gain=float(globals().get("PT_PAIR_MIN_GAIN", 0.002)),
            global_fit_kwargs=POSTPROCESS_FIT_KWARGS,
            bias_values=globals().get("PT_PAIR_BIAS_VALUES", (-0.30, -0.15, 0.0, 0.15, 0.30)),
            max_margin=float(globals().get("PT_PAIR_MAX_MARGIN", 0.75)),
        )
        globals().update(_postprocess_result)
    print("Global parameters retained:", POSTPROCESS_CONFIG["parameters"])
    print("Portuguese pair policy:", POSTPROCESS_CONFIG.get("portuguese_pair", {"enabled": False}))


### Xuất CSV selection và gói dự đoán nhỏ

`submission_selection.csv` là kết quả đã chọn sau các điều kiện kiểm tra.
`submission_selection_global.csv` giữ kết quả trước bước PT;
`submission_selection_raw.csv` giữ dự đoán neural thô trong artifacts.
Nếu bước PT không đạt điều kiện, hai file selection và selection_global giống nhau.

Tải **`/content/submission_selection.csv`** để nộp và giữ thêm
**`/content/selection_predictions.zip`** để thử hậu xử lý lần sau mà không train lại.
ZIP này không chứa weights. `submission.csv` hiện có cùng nội dung selection;
nếu bật refit thì bước sau mới cập nhật nó bằng dự đoán model refit.


In [ ]:
def validate_submission(frame, expected_test, template):
    assert list(frame.columns) == ["id", "label"], "Sai cot submission."
    assert len(frame) == len(expected_test) == 6000, "Sai so dong."
    assert frame["id"].is_unique, "ID trung."
    assert frame.notna().all().all(), "Submission co gia tri thieu."
    assert set(frame["id"]) == set(expected_test["id"]), "Thieu/thua ID."
    assert frame["id"].tolist() == template["id"].tolist(), "Sai thu tu sample."
    assert frame["label"].isin(LABELS).all(), "Nhan khong hop le."

def make_submission(predicted, destination):
    assert len(predicted) == len(test_df)
    predictions = pd.DataFrame({"id": test_df["id"].to_numpy(),
                                "label": [ID2LABEL[int(i)] for i in predicted]})
    result = sample_df[["id"]].merge(predictions, on="id", how="left", sort=False, validate="one_to_one")
    validate_submission(result, test_df, sample_df)
    result.to_csv(destination, index=False, encoding="utf-8")
    validate_submission(pd.read_csv(destination, dtype=str, keep_default_na=False), test_df, sample_df)
    return result

# Luu selection CSV truoc full refit de co ket qua neu runtime bi ngat.
make_submission(global_test_pred, ARTIFACT_DIR / "submission_selection_global.csv")
shutil.copy2(ARTIFACT_DIR / "submission_selection_global.csv", "/content/submission_selection_global.csv")
make_submission(test_logits.argmax(axis=1), ARTIFACT_DIR / "submission_selection_raw.csv")
SUBMISSION_PATH = Path("/content/submission.csv")
submission = make_submission(final_test_pred, SUBMISSION_PATH)
shutil.copy2(SUBMISSION_PATH, ARTIFACT_DIR / "submission_selection.csv")
shutil.copy2(SUBMISSION_PATH, "/content/submission_selection.csv")

final_report = pd.DataFrame(classification_report(
    valid_true, final_valid_pred, labels=np.arange(len(LABELS)), target_names=LABELS,
    output_dict=True, zero_division=0,
)).T
print("Selection fitted validation report (NOT an independent score):")
display(final_report)
final_report.to_csv(ARTIFACT_DIR / "validation_final_fitted_report.csv")
final_language_rows = []
for language in LANGUAGES:
    mask = valid_df["language"].eq(language).to_numpy()
    scores = f1_score(valid_true[mask], final_valid_pred[mask], labels=np.arange(len(LABELS)),
                      average=None, zero_division=0)
    row = {"language": language, "rows": int(mask.sum()), "macro_f1": float(scores.mean())}
    row.update({f"f1_{label}": float(s) for label, s in zip(LABELS, scores)})
    final_language_rows.append(row)
pd.DataFrame(final_language_rows).to_csv(ARTIFACT_DIR / "validation_final_by_language.csv", index=False)
final_matrix = confusion_matrix(valid_true, final_valid_pred, labels=np.arange(len(LABELS)))
pd.DataFrame(final_matrix, index=LABELS, columns=LABELS).to_csv(ARTIFACT_DIR / "validation_final_confusion.csv")
final_errors = build_prediction_audit(
    valid_df, valid_true, final_valid_scores, valid_dataset.raw_lengths,
    budget, LABELS, stage="final",
)
final_groups, final_lengths = summarize_prediction_audit(final_errors, stage="final")
final_groups.to_csv(ARTIFACT_DIR / "validation_errors_by_group_final.csv", index=False)
final_lengths.to_csv(ARTIFACT_DIR / "validation_errors_by_length_final.csv", index=False)
final_errors.to_csv(ARTIFACT_DIR / "validation_prediction_audit_final.csv", index=False)
final_errors.loc[valid_true != final_valid_pred].to_csv(ARTIFACT_DIR / "validation_errors_final.csv", index=False)
final_valid_rows = valid_df[["id", "language", "label"]].copy()
final_valid_rows["predicted_label"] = [ID2LABEL[int(i)] for i in final_valid_pred]
final_valid_rows.to_csv(ARTIFACT_DIR / "validation_final_predictions.csv", index=False)
np.save(ARTIFACT_DIR / "test_final_probabilities.npy", final_test_proba)
(ARTIFACT_DIR / "postprocess.json").write_text(
    json.dumps(POSTPROCESS_CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")
(ARTIFACT_DIR / "postprocess_selection.json").write_text(
    json.dumps(POSTPROCESS_SELECTION_REPORT, indent=2, ensure_ascii=False), encoding="utf-8")
oof_report = valid_df[["id", "language", "label"]].copy()
oof_report["predicted_label"] = [ID2LABEL[int(i)] for i in postprocess_oof_pred]
oof_report.to_csv(ARTIFACT_DIR / "validation_postprocess_crossfit.csv", index=False)
display(submission.head())
display(submission["label"].value_counts().reindex(LABELS, fill_value=0))
print("Submission:", SUBMISSION_PATH)
print("SHA256:", hashlib.sha256(SUBMISSION_PATH.read_bytes()).hexdigest())

print("Co the submit ngay: /content/submission_selection.csv (Files panel > Refresh > Download).")

"""Append to selection CSV export: save a small bundle for future CPU tuning."""

# These predictions are captured BEFORE any full-data refit replaces test_logits.
np.save(ARTIFACT_DIR / "selection_test_logits.npy", np.asarray(test_logits, dtype=np.float32))
np.save(ARTIFACT_DIR / "selection_test_probabilities.npy", np.asarray(final_test_proba))
if valid_lexical_scores is not None:
    np.save(ARTIFACT_DIR / "validation_lexical_scores.npy", valid_lexical_scores)
if test_lexical_scores is not None:
    np.save(ARTIFACT_DIR / "selection_test_lexical_scores.npy", test_lexical_scores)

pt_report = POSTPROCESS_SELECTION_REPORT.get("portuguese_pair", {
    "enabled": False, "reason": "disabled_by_configuration",
})
(ARTIFACT_DIR / "pt_pair_tuning.json").write_text(
    json.dumps(pt_report, indent=2, ensure_ascii=False), encoding="utf-8",
)
(ARTIFACT_DIR / "global_postprocess.json").write_text(
    json.dumps(GLOBAL_POSTPROCESS_CONFIG, indent=2, ensure_ascii=False), encoding="utf-8",
)
raw_macro_current = float(macro_f1(valid_true, valid_logits.argmax(axis=1)))
selection_comparison = {
    "stage": "selection",
    "v3_raw_validation_reference": 0.9729325186,
    "v4_raw_validation_user_reported": 0.969274,
    "current_raw_validation_macro_f1": raw_macro_current,
    "raw_delta_vs_v3": raw_macro_current - 0.9729325186,
    "global_crossfit_macro_f1": GLOBAL_POSTPROCESS_SELECTION_REPORT["cross_fitted_postprocess_macro_f1"],
    "global_gate_passed": bool(GLOBAL_POSTPROCESS_CONFIG["enabled"]),
    "portuguese_pair_enabled": bool(pt_report["enabled"]),
    "portuguese_pair_nested_gain": pt_report.get("nested_gain"),
    "final_fitted_validation_macro_f1": float(macro_f1(valid_true, final_valid_pred)),
    "note": "Fitted scores are diagnostics. Checkpoint selection used this validation; no Kaggle gain is guaranteed.",
}
(ARTIFACT_DIR / "selection_comparison.json").write_text(
    json.dumps(selection_comparison, indent=2, ensure_ascii=False), encoding="utf-8",
)
if "portuguese_pair_baseline_oof_pred" in globals():
    pt_oof = valid_df[["id", "language", "label"]].copy()
    pt_oof["baseline_prediction"] = [ID2LABEL[int(i)] for i in portuguese_pair_baseline_oof_pred]
    pt_oof["candidate_prediction"] = [ID2LABEL[int(i)] for i in portuguese_pair_candidate_oof_pred]
    pt_oof.to_csv(ARTIFACT_DIR / "validation_portuguese_pair_oof.csv", index=False)

# Use an explicit allowlist: never include checkpoints, optimizer state, or a
# final-refit array whose rows came from a different model than validation.
selection_bundle_names = [
    "validation_logits.npy", "validation_ids.csv", "test_ids.csv", "split.csv",
    "selection_test_logits.npy", "selection_test_probabilities.npy",
    "validation_lexical_scores.npy", "selection_test_lexical_scores.npy",
    "submission_selection.csv", "submission_selection_raw.csv", "submission_selection_global.csv",
    "postprocess.json", "global_postprocess.json", "postprocess_selection.json",
    "pt_pair_tuning.json", "selection_comparison.json", "validation_predictions.csv",
    "validation_errors_raw.csv", "validation_portuguese_pair_oof.csv",
    "model_provenance.json", "model_compatibility.json",
]
required_selection_files = {
    "validation_logits.npy", "validation_ids.csv", "test_ids.csv", "split.csv",
    "selection_test_logits.npy", "submission_selection.csv", "submission_selection_raw.csv",
    "submission_selection_global.csv", "postprocess.json", "global_postprocess.json",
    "selection_comparison.json",
}
missing_selection_files = sorted(name for name in required_selection_files if not (ARTIFACT_DIR / name).is_file())
if missing_selection_files:
    raise FileNotFoundError(f"Selection bundle missing: {missing_selection_files}")
available_selection_files = [name for name in selection_bundle_names if (ARTIFACT_DIR / name).is_file()]
selection_manifest = {
    "schema_version": 1, "stage": "selection_only", "contains_model_weights": False,
    "notebook_version": NOTEBOOK_VERSION, "run_id": RUN_ID,
    "model_preset": MODEL_PRESET, "model_provenance": MODEL_PROVENANCE,
    "wolf_loading_mode": WOLF_LOADING_MODE if MODEL_PRESET == "wolf" else None,
    "labels": list(LABELS), "label2id": LABEL2ID,
    "data_sha256": DATA_HASHES, "split_seed": SPLIT_SEED,
    "fit_rows": len(fit_df), "validation_rows": len(valid_df), "test_rows": len(test_df),
    "best_validation_macro_f1": float(trainer.state.best_metric),
    "selected_epoch": int(selected_checkpoint_epoch(trainer.state)),
    "lexical_enabled": bool(valid_lexical_scores is not None),
    "test_logits_file": "selection_test_logits.npy",
    "validation_logits_file": "validation_logits.npy",
    "file_sha256": {name: hashlib.sha256((ARTIFACT_DIR / name).read_bytes()).hexdigest()
                    for name in available_selection_files},
}
(ARTIFACT_DIR / "selection_manifest.json").write_text(
    json.dumps(selection_manifest, indent=2, ensure_ascii=False), encoding="utf-8",
)
SELECTION_PREDICTIONS_PATH = Path("/content/selection_predictions.zip")
with zipfile.ZipFile(SELECTION_PREDICTIONS_PATH, "w", compression=zipfile.ZIP_DEFLATED) as packed:
    for name in available_selection_files + ["selection_manifest.json"]:
        packed.write(ARTIFACT_DIR / name, arcname=name)
with zipfile.ZipFile(SELECTION_PREDICTIONS_PATH) as packed:
    for name, digest in selection_manifest["file_sha256"].items():
        assert hashlib.sha256(packed.read(name)).hexdigest() == digest
    assert packed.read("submission_selection.csv") == (ARTIFACT_DIR / "submission_selection.csv").read_bytes()
print("Raw Macro F1:", round(raw_macro_current, 6), "| V3 reference: 0.972933")
print("Portuguese pair enabled:", pt_report["enabled"], "| nested gain:", pt_report.get("nested_gain"))
print("Download small predictions bundle:", SELECTION_PREDICTIONS_PATH,
      f"({SELECTION_PREDICTIONS_PATH.stat().st_size / 2**20:.2f} MiB)")


## 11. Refit tùy chọn trên toàn bộ dữ liệu có nhãn

Mặc định **`USE_FULL_TRAIN=False`**: bước này chỉ lưu metadata của selection và không
train thêm. Muốn refit sau khi xem kết quả, đặt `USE_FULL_TRAIN=True` trong một cell
ngay trước bước này, giữ nguyên các cấu hình đã dùng cho selection.

Refit khởi tạo mới từ pretrained, train 14.000 dòng trong số epoch checkpoint tốt
nhất đã chọn. Không đánh giá trên holdout cũ đã học, không chọn lại bias.
Hiệu chỉnh global/PT được giữ nguyên và áp dụng theo đúng language/ID của test.
Điểm raw/cross-fit trước đó thuộc selection; refit không bảo đảm tăng điểm test.


In [ ]:
"""Notebook cell source: refit a fresh model on all labelled training rows.

Paste after the selection submission/artifacts cell. This source intentionally
depends on the notebook namespace and must not be run as a standalone script.
Validation predictions below always belong to the held-out selection stage.
"""

# Capture the selection state before replacing trainer/model. These Python and
# NumPy objects contain no live GPU model or optimizer references.
SELECTED_EPOCH = int(selected_checkpoint_epoch(trainer.state))
SELECTION_TRAINING_LOG = json.loads(json.dumps(trainer.state.log_history))
SELECTION_TRAINING_ARGUMENTS = json.loads(training_args.to_json_string())
SELECTION_TRAIN_METRICS = dict(train_result.metrics)
FROZEN_POSTPROCESS_CONFIG = json.loads(json.dumps(POSTPROCESS_CONFIG))
SELECTION_REPORT = {
    "training_rows": int(len(fit_df)),
    "validation_rows": int(len(valid_df)),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_macro_f1": float(trainer.state.best_metric),
    "selected_epoch": SELECTED_EPOCH,
    "adapter": str(ADAPTER_DIR.relative_to(ARTIFACT_DIR)),
    "lexical": "selection_lexical.joblib" if ENABLE_LEXICAL_BRANCH else None,
    "postprocess_selection": json.loads(json.dumps(POSTPROCESS_SELECTION_REPORT)),
}
(ARTIFACT_DIR / "selection_report.json").write_text(
    json.dumps(SELECTION_REPORT, indent=2, ensure_ascii=False), encoding="utf-8",
)
(ARTIFACT_DIR / "selection_training_log.json").write_text(
    json.dumps(SELECTION_TRAINING_LOG, indent=2), encoding="utf-8",
)
(ARTIFACT_DIR / "selection_training_arguments.json").write_text(
    json.dumps(SELECTION_TRAINING_ARGUMENTS, indent=2), encoding="utf-8",
)
(ARTIFACT_DIR / "selection_train_metrics.json").write_text(
    json.dumps(SELECTION_TRAIN_METRICS, indent=2), encoding="utf-8",
)

# Preserve the 90/10 model's predictions even when full-train predictions later
# replace the notebook's active test variables and submission.csv.
selection_test_logits = np.array(test_logits, dtype=np.float32, copy=True)
selection_test_probabilities = np.array(final_test_proba, copy=True)
np.save(ARTIFACT_DIR / "selection_test_logits.npy", selection_test_logits)
np.save(ARTIFACT_DIR / "selection_test_probabilities.npy", selection_test_probabilities)
if test_lexical_scores is not None:
    np.save(ARTIFACT_DIR / "selection_test_lexical_scores.npy", test_lexical_scores)

FINAL_FIT_REPORT = {
    "enabled": bool(USE_FULL_TRAIN),
    "completed": not USE_FULL_TRAIN,
    "training_rows": int(len(fit_df)),
    "epochs": SELECTED_EPOCH,
    "epoch_source": "holdout_best_checkpoint",
    "active_adapter": SELECTION_REPORT["adapter"],
    "active_lexical": SELECTION_REPORT["lexical"],
    "postprocess_frozen": True,
    "validation_evaluated_after_refit": False,
    "full_train_validation_macro_f1": None,
    "note": (
        "Validation and cross-fit scores refer only to the selection model. "
        "A refit on all labelled rows has no untouched validation score; "
        "its frozen temperatures/biases may transfer imperfectly."
    ),
}

if USE_FULL_TRAIN:
    refit_epochs = float(SELECTED_EPOCH if FULL_TRAIN_EPOCHS is None else FULL_TRAIN_EPOCHS)
    if not np.isfinite(refit_epochs) or refit_epochs <= 0:
        raise ValueError("FULL_TRAIN_EPOCHS phai la so duong hoac None.")
    assert len(train_df) == 14000
    assert set(train_df["id"]).isdisjoint(set(test_df["id"]))
    full_train_df = train_df.reset_index(drop=True).copy()
    full_train_dataset = PromptDataset(full_train_df, labelled=True)
    full_weight_values = compute_class_weights(
        full_train_df["label_id"].to_numpy(), len(LABELS),
        power=CLASS_WEIGHT_POWER, enabled=USE_CLASS_WEIGHTS,
    )
    full_class_weights = torch.tensor(full_weight_values, dtype=torch.float32)
    full_weighted_focal_loss = make_weighted_focal_loss(full_class_weights, FOCAL_GAMMA)
    full_train_df[["id", "language", "label"]].to_csv(
        ARTIFACT_DIR / "full_train_rows.csv", index=False,
    )
    FINAL_FIT_REPORT.update({
        "training_rows": int(len(full_train_df)),
        "epochs": refit_epochs,
        "epoch_source": "holdout_best_checkpoint" if FULL_TRAIN_EPOCHS is None else "fixed_config_override",
        "class_weights": np.asarray(full_weight_values).tolist(),
    })
    print(
        "Full refit:", len(full_train_df), "rows | epochs =", refit_epochs,
        "| fresh backbone/head (adapter neu bat) | khong evaluate/tune tren validation.",
    )

    # Release Accelerator's references as well as Trainer's model, optimizer and
    # callbacks before allocating another backbone on the same GPU.
    trainer.accelerator.free_memory()
    del trainer, model, train_result
    gc.collect()
    torch.cuda.empty_cache()
    set_seed(SEED)
    model = build_model()
    training_args = TrainingArguments(
        output_dir=str(RUN_DIR / "full_train"),
        run_name=RUN_NAME + "_full_train",
        num_train_epochs=refit_epochs,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        max_grad_norm=1.0,
        fp16=not USE_BF16,
        bf16=USE_BF16,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="no",
        save_strategy="no",
        load_best_model_at_end=False,
        logging_strategy="steps",
        logging_steps=25,
        label_names=["labels"],
        remove_unused_columns=True,
        eval_accumulation_steps=16,
        dataloader_num_workers=0,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
        push_to_hub=False,
        torch_compile=False,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=full_train_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_loss_func=full_weighted_focal_loss,
    )
    trainer.model_accepts_loss_kwargs = False
    # Explicitly start from the factory's fresh model, never resume the 90/10 fit.
    train_result = trainer.train(resume_from_checkpoint=False)
    trainer.save_metrics("train", train_result.metrics)
    trainer.save_state()
    ADAPTER_DIR = ARTIFACT_DIR / ("final_model" if MODEL_PRESET == "wolf" else "final_adapter")
    trainer.model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    trainer.model.config.to_json_file(str(ARTIFACT_DIR / "base_config.json"))

    test_result = trainer.predict(test_dataset, metric_key_prefix="test")
    test_logits = test_result.predictions
    if isinstance(test_logits, tuple):
        test_logits = test_logits[0]
    test_logits = np.asarray(test_logits, dtype=np.float32)
    if test_logits.shape != (len(test_df), len(LABELS)) or not np.isfinite(test_logits).all():
        raise ValueError("Full-train test logits sai shape hoac co NaN/Inf.")

    # Refit the vocabulary, IDF and classifiers on labelled train.csv only.
    # No validation or test scores/labels enter model or postprocessor selection.
    test_lexical_scores = None
    if ENABLE_LEXICAL_BRANCH:
        lexical_bundle = fit_lexical_branch(full_train_df)
        test_lexical_scores = lexical_predict(lexical_bundle, test_df)
        joblib.dump(lexical_bundle, ARTIFACT_DIR / "final_lexical.joblib", compress=3)
        np.save(ARTIFACT_DIR / "test_lexical_scores.npy", test_lexical_scores)
        FINAL_FIT_REPORT["active_lexical"] = "final_lexical.joblib"

    # Apply exactly the holdout-selected configuration; no postprocess fit call.
    final_test_scores = apply_selected_postprocess(
        test_logits, test_lexical_scores, FROZEN_POSTPROCESS_CONFIG,
        test_df["language"].astype(str).to_numpy(),
    )
    final_test_proba = softmax_from_log_scores(final_test_scores)
    final_test_pred = final_test_proba.argmax(axis=1)
    assert POSTPROCESS_CONFIG == FROZEN_POSTPROCESS_CONFIG
    np.testing.assert_allclose(final_test_proba.sum(axis=1), 1.0, atol=1e-6)
    assert np.isfinite(final_test_proba).all()
    np.save(ARTIFACT_DIR / "test_logits.npy", test_logits)
    np.save(ARTIFACT_DIR / "test_final_probabilities.npy", final_test_proba)
    FINAL_FIT_REPORT.update({
        "completed": True,
        "active_adapter": str(ADAPTER_DIR.relative_to(ARTIFACT_DIR)),
        "train_metrics": dict(train_result.metrics),
        "optimizer_steps": int(trainer.state.global_step),
    })
else:
    print("USE_FULL_TRAIN=False: giu model va submission tu holdout selection.")

(ARTIFACT_DIR / "final_fit_report.json").write_text(
    json.dumps(FINAL_FIT_REPORT, indent=2, ensure_ascii=False), encoding="utf-8",
)
print("Active model/adapter:", FINAL_FIT_REPORT["active_adapter"])
print("Active lexical model:", FINAL_FIT_REPORT["active_lexical"])
print("Validation reports van thuoc selection stage; chay cell xuat CSV tiep theo.")


In [ ]:
# Xuat du doan cua active model; final_valid_* van la selection predictions.
submission = make_submission(final_test_pred, SUBMISSION_PATH)
shutil.copy2(SUBMISSION_PATH, ARTIFACT_DIR / "submission.csv")
make_submission(test_logits.argmax(axis=1), ARTIFACT_DIR / "submission_raw.csv")
np.save(ARTIFACT_DIR / "test_final_probabilities.npy", final_test_proba)
print("Submission:", SUBMISSION_PATH, "| rows:", len(submission))
print("Weights:", FINAL_FIT_REPORT["active_adapter"])
print("SHA256:", hashlib.sha256(SUBMISSION_PATH.read_bytes()).hexdigest())
display(submission["label"].value_counts().reindex(LABELS, fill_value=0))


## 12. Lưu cấu hình và đóng gói

ZIP chứa selection và final weights: full model cho Wolf, adapter + heads cho DistilBERT.
Vì Wolf lưu toàn bộ weights, ZIP lớn hơn bản QLoRA. Bao gồm tokenizer, lexical joblib của từng
stage, postprocess.json, logits/ID, split, cấu hình và CSV. Đọc active_weights/active_lexical
trong run_config.json để tái lập submission; không ghép nhầm các stage.
DistilBERT cần tải lại đúng pretrained revision khi nạp adapter. Checkpoint chứa optimizer
ở RUN_DIR không nằm trong ZIP. Validation chỉ thuộc selection; refit dùng đủ 14.000 mẫu.


In [ ]:
# Re-read and synchronize the final CSV immediately before making the ZIP.
validate_submission(pd.read_csv(SUBMISSION_PATH, dtype=str, keep_default_na=False), test_df, sample_df)
shutil.copy2(SUBMISSION_PATH, ARTIFACT_DIR / "submission.csv")
run_config = {
    "team": "RMIT_SGS_Sigma69_C1",
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_source": MODEL_SOURCE,
    "requested_revision": MODEL_REVISION,
    "resolved_model_revision": RESOLVED_MODEL_REVISION,
    "labels": LABELS,
    "label2id": LABEL2ID,
    "seed": SEED,
    "split_seed": SPLIT_SEED,
    "notebook_version": NOTEBOOK_VERSION,
    "model_preset": MODEL_PRESET,
    "model_provenance": MODEL_PROVENANCE,
    "wolf_loading_mode": WOLF_LOADING_MODE if MODEL_PRESET == "wolf" else None,
    "v4_best_raw_validation_user_reported": 0.969274,
    "selection": SELECTION_REPORT,
    "full_refit": FINAL_FIT_REPORT,
    "active_weights": FINAL_FIT_REPORT["active_adapter"],
    "active_lexical": FINAL_FIT_REPORT["active_lexical"],
    "weights_format": "full_model" if MODEL_PRESET == "wolf" else "peft_adapter",
    "modernbert_reference_compile": False if MODEL_PRESET == "wolf" else None,
    "validation_size": VAL_SIZE,
    "training_rows": FINAL_FIT_REPORT["training_rows"],
    "validation_rows": len(valid_df),
    "test_rows": len(test_df),
    "max_length": MAX_LENGTH,
    "long_text_mode": LONG_TEXT_MODE,
    "epochs_requested": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
    "class_weights": FINAL_FIT_REPORT.get("class_weights", weight_values.tolist()),
    "class_weight_power": power,
    "postprocess": POSTPROCESS_CONFIG,
    "lexical": {"enabled": ENABLE_LEXICAL_BRANCH, "analyzer": "char_wb",
                "ngram_range": [2, 6], "C": LEXICAL_C, "class_weight": "balanced",
                "min_df": 2, "sublinear_tf": True, "max_features": LEXICAL_MAX_FEATURES,
                "global_weight": LEXICAL_GLOBAL_WEIGHT},
    "loss": "weighted_focal_mean_over_samples" if FOCAL_GAMMA else "weighted_cross_entropy_mean_over_samples",
    "focal_gamma": FOCAL_GAMMA,
    "quantization": {
        "load_in_4bit": True, "quant_type": "nf4", "double_quant": True,
        "compute_dtype": str(COMPUTE_DTYPE), "skip_modules": HEAD_MODULES,
    } if MODEL_PRESET == "distilbert" else None,
    "lora": {
        "rank": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGETS, "modules_to_save": HEAD_MODULES,
    } if MODEL_PRESET == "distilbert" else None,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_capability": list(GPU_CAPABILITY),
    "package_versions": PACKAGE_VERSIONS,
    "data_sha256": DATA_HASHES,
    "best_checkpoint": SELECTION_REPORT["best_checkpoint"],
    "best_validation_macro_f1": SELECTION_REPORT["best_validation_macro_f1"],
    "baseline_validation_macro_f1": 0.8863657407407408,
    "baseline_public_score_user_reported": 0.87,
    "v3_wolf_raw_validation_macro_f1": 0.9729325186,
    "v3_wolf_public_score_user_reported": 0.97,
    "v3_archive_sha256": "cea7a8332988234fcf2a232c3b25d9256964519cd2c7a2f90d11131a6d02e9a5",
    "submission_sha256": hashlib.sha256(SUBMISSION_PATH.read_bytes()).hexdigest(),
}
(ARTIFACT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2, ensure_ascii=False), encoding="utf-8",
)
(ARTIFACT_DIR / "label_mapping.json").write_text(
    json.dumps({"label2id": LABEL2ID, "id2label": ID2LABEL}, indent=2), encoding="utf-8",
)
(ARTIFACT_DIR / "validation_metrics.json").write_text(
    json.dumps({"stage": "selection_only", "metrics": valid_result.metrics}, indent=2), encoding="utf-8",
)
(ARTIFACT_DIR / "training_arguments.json").write_text(
    training_args.to_json_string(), encoding="utf-8",
)
(ARTIFACT_DIR / "training_log.json").write_text(
    json.dumps(trainer.state.log_history, indent=2), encoding="utf-8",
)
(ARTIFACT_DIR / "requirements-used.txt").write_text(
    "\n".join(f"{name}=={release}" for name, release in PACKAGE_VERSIONS.items()) + "\n",
    encoding="utf-8",
)
ARCHIVE_PATH = Path(shutil.make_archive(
    str(RUN_DIR / "rmit_c1_artifacts"), "zip", root_dir=str(ARTIFACT_DIR),
))
with zipfile.ZipFile(ARCHIVE_PATH) as packed:
    packed_csv = packed.read("submission.csv")
    packed_config = json.loads(packed.read("run_config.json"))
    packed_hash = hashlib.sha256(packed_csv).hexdigest()
    assert packed_hash == run_config["submission_sha256"] == packed_config["submission_sha256"]
    assert packed_csv == SUBMISSION_PATH.read_bytes(), "CSV trong ZIP khac file hien tai."
print("ZIP/CSV SHA256 da khop:", packed_hash)
print("Artifacts:", ARTIFACT_DIR)
print("ZIP:", ARCHIVE_PATH, f"({ARCHIVE_PATH.stat().st_size / 2**20:.1f} MiB)")
PUBLIC_ARCHIVE_PATH = Path("/content/rmit_c1_artifacts.zip")
if ARCHIVE_PATH.resolve() != PUBLIC_ARCHIVE_PATH.resolve():
    shutil.copy2(ARCHIVE_PATH, PUBLIC_ARCHIVE_PATH)
print("Files panel download:", PUBLIC_ARCHIVE_PATH)


## 13. Tải kết quả

Mở Files bên trái Colab → Refresh. Tải `submission_selection.csv` và
`selection_predictions.zip` để giữ kết quả selection cùng logits/ID.
`submission.csv` là file của model đang dùng (selection nếu không refit, final nếu
đã refit). `rmit_c1_artifacts.zip` chứa model và báo cáo nên lớn hơn nhiều.
Các cell dưới chỉ in đường dẫn; đặt cờ `True` nếu muốn thử tải tự động.


In [ ]:
DOWNLOAD_CSV_NOW = False
print("Files panel: /content/submission.csv")
print("Size:", SUBMISSION_PATH.stat().st_size, "bytes")
if DOWNLOAD_CSV_NOW:
    from google.colab import files
    files.download(str(SUBMISSION_PATH))

print("Giu them ZIP nho de tinh chinh lan sau:", SELECTION_PREDICTIONS_PATH)


In [ ]:
DOWNLOAD_ZIP_NOW = False
print("Files panel:", PUBLIC_ARCHIVE_PATH)
if DOWNLOAD_ZIP_NOW:
    from google.colab import files
    files.download(str(PUBLIC_ARCHIVE_PATH))


## Nguồn và cách đọc kết quả

- [Cuộc thi](https://www.kaggle.com/competitions/rmit-hackathon-2026/overview).
- [Wolf trên Kaggle](https://www.kaggle.com/models/aisuko/wolf-defender-prompt-injection/transformers/default/1).
- [AutoTokenizer 4.57.6](https://github.com/huggingface/transformers/blob/v4.57.6/src/transformers/models/auto/tokenization_auto.py).
- [Nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).

`selection_comparison.json` tách raw Macro F1 hiện tại, mốc V3/V4 và gain hiệu chỉnh.
`pt_pair_tuning.json` lưu điều kiện bật/tắt cùng các fold. Cross-fit chỉ đánh giá
phần hậu xử lý; model vẫn từng dùng holdout này để chọn checkpoint.

Nạp lại inference phải dùng `apply_selected_postprocess` với config đầy đủ và
language đúng thứ tự ID. Trường `enabled` ở gốc config chỉ mô tả global policy;
hiệu chỉnh PT nằm trong `portuguese_pair`. Refit áp dụng cả hai mà không fit lại.
`selection_predictions.zip` chỉ giữ dự đoán selection; model nằm trong ZIP artifacts lớn.
Bản V3 đã chạy và source V4 được giữ riêng trong `notebooks/archive/`.
